# ⏳ 32 · Cell dynamics: pseudotime, RNA velocity and optimal transport
### Reconstructing processes from snapshots — diffusion pseudotime, trajectory skeletons, the kinetics of splicing, RNA velocity and where it fails, Sinkhorn optimal transport between time points and Markov-chain fate probabilities, all graded against a known truth

**BiologyToML** · Level 2 · Research frontier · Part G — Cells and tissues · Module 32 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/32_cell_dynamics_velocity.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (sparse matrices, eigensolvers, minimum spanning trees), pandas, Matplotlib, scikit-learn (nearest neighbours, k-means, AUROC) |
| **Prerequisites** | Level 1 (especially Modules 04, 14 and 19–20) and [Module 30](30_single_cell_rnaseq.ipynb) (single-cell pipeline: normalisation, PCA, k-NN graphs); comfortable with NumPy |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [04 · Calculus and optimisation](04_calculus_and_optimization.ipynb), [06 · Linear regression](06_linear_regression_gwas.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [17 · Bayesian networks: inference and learning](17_bayesian_networks_inference_learning.ipynb), [19 · HMMs I](19_hmm_inference.ipynb), [20 · HMMs II](20_hmm_learning.ipynb), [30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb) |
| **Research frontier covered** | trajectory inference and pseudotime, RNA velocity, optimal-transport analysis of time courses, cell-fate probabilities — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** with the toolkit's `simulate_differentiation`: 2,000 cells sampled on four days from a
> process in which progenitors split into two fates (A and B). Every gene follows the same transcription–splicing–
> degradation equations that RNA-velocity methods assume. The hidden truth includes each cell's latent time and
> eventual fate, each gene's programme and kinetic rates (α, β, γ) and the true velocity of every gene in every cell.
> Because the simulator obeys the model exactly, methods look better here than on real data; Section 5 provokes
> failures on purpose and Section 8 explains what changes with real data.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why single-cell data are snapshots, what spliced and unspliced reads measure, and why the order, direction and outcome of a process must be *inferred*.
2. **Build** a diffusion map from a k-NN graph (a Markov chain on cells), **compute** diffusion pseudotime from a root cell and **check** it against the true latent time.
3. **Summarise** a trajectory as a tree over clusters (minimum spanning tree, PAGA-like connectivity) and **verify** that it matches the true lineage.
4. **Solve** the transcription–splicing–degradation equations for one gene, **read** phase portraits, **estimate** steady-state RNA velocity and **project** it onto an embedding, scoring every step against the truth.
5. **Provoke** and **diagnose** failures of RNA velocity (genes far from steady state, lineage-specific kinetics).
6. **Derive** and **implement** entropic optimal transport (Sinkhorn) between sampling days, and **compute** ancestors, descendants and fate probabilities.
7. **Show** that early fate is unpredictable when it is not written in the transcriptome (AUROC ≈ 0.5), and detectable when progenitors are primed.
8. **Compute** fate probabilities as absorption probabilities of a velocity-directed Markov chain (the CellRank idea) and **compare** them with optimal transport.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Snapshots of a process | spliced and unspliced counts; the SIMULATED time course | 15 min |
| 2 · Pseudotime with diffusion maps | random walk on the k-NN graph, eigenvectors, DPT; 🧪 vs true time | 25 min |
| 3 · A trajectory skeleton | clusters + minimum spanning tree, PAGA-like graph; 🧪 vs true lineage | 15 min |
| 4 · RNA velocity | kinetic equations, phase portraits, γ′, velocity graph; 🧪 vs true velocity | 35 min |
| 5 · When velocity fails | genes far from steady state; lineage-specific kinetics | 15 min |
| 6 · Optimal transport between days | Kantorovich, Sinkhorn, ancestors, fates; 🧪 AUROC, priming | 30 min |
| 7 · Markov-chain fate mapping | absorbing chains, CellRank idea; 🧪 vs truth and vs OT | 20 min |
| 8 · 🧭 From toy to real data | scVelo, CellRank, moscot, lineage tracing | 10 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

A blood stem cell becomes a neutrophil or a monocyte; a pancreatic progenitor becomes an α or a β cell; a fibroblast,
pushed by four transcription factors, sometimes becomes a stem cell. Developmental biology is about **processes** — the
order of events, their direction, and the moment a cell's fate is decided. Single-cell RNA-seq measures thousands of
cells, but it **destroys each cell it measures**: we never see the same cell twice. A data set is a collection of
snapshots, like photographs of a crowd in which nobody is tracked.

Three ideas let researchers reconstruct processes from snapshots anyway, and each is a landmark:

1. **Pseudotime.** If differentiation is gradual, cells caught at different moments form a continuous path in
   expression space; ordering cells along the path gives a "pseudo" time (Monocle, Trapnell *et al.* 2014; diffusion
   pseudotime, Haghverdi *et al.* 2016).
2. **RNA velocity.** Newly made RNA still contains introns. Comparing unspliced (new) with spliced (mature) mRNA tells
   whether a gene is being switched on or off *right now*, which gives each cell an arrow pointing to its near future
   (La Manno *et al.* 2018; scVelo, Bergen *et al.* 2020).
3. **Optimal transport.** In a time course, the cells of day 2 descend from the cells of day 1. Finding the cheapest
   way to "move" one population onto the next infers ancestors and descendants (Waddington-OT, Schiebinger *et al.*
   2019, on fibroblast-to-stem-cell reprogramming).

And one sobering result: when Weinreb *et al.* (2020) followed blood progenitors with heritable DNA barcodes (true
lineage tracing), the fate of a progenitor was only partly predictable from its transcriptome. Some of what decides a
fate may simply not be visible in the RNA.

Our SIMULATED data mimic a typical experiment: progenitors that split into two fates, sampled on four days. Every gene
follows the textbook kinetics, and we know each cell's true time and fate — so every method can be graded.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse, stats
from scipy.sparse.csgraph import breadth_first_order, connected_components, minimum_spanning_tree
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_differentiation` generates the time course with its hidden truth (latent time, fate, kinetic rates, true velocity); `log_normalize` and `pca` are the standard preprocessing of [Module 30](30_single_cell_rnaseq.ipynb); `plot_embedding` draws 2-D maps of cells. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_differentiation(rng: np.random.Generator, n_cells: int = 2000, n_genes: int = 200,
                             branch_time: float = 0.35, n_days: int = 4, depth: float = 25.0,
                             priming: float = 0.0) -> dict:
    """Simulate a differentiation time course: progenitors split into fate A or fate B.

    Every gene follows the transcription kinetics used by RNA-velocity methods,
    du/dt = α(t) − β·u (unspliced pre-mRNA) and ds/dt = β·u − γ·s (spliced mRNA), where the
    transcription rate α(t) switches on and off along each lineage.  Gene programmes:
    progenitor genes (switch off), fate-A genes, fate-B genes, transient genes and
    housekeeping genes.  Cells are collected on ``n_days`` sampling days (snapshots overlap
    in latent time, as in a real time course).  ``priming > 0`` lets a few fate genes
    differ slightly *before* the branch point (early fate bias).

    Returns a dict with ``spliced`` and ``unspliced`` counts (cells × genes), ``obs`` (true
    latent time, sampling day, state and eventual fate), ``var`` (programme and kinetic
    rates of every gene) and the true ``velocity`` ds/dt of every cell.
    """
    grid = np.linspace(0.0, 1.0, 801)
    dt = grid[1] - grid[0]
    program = rng.choice(["progenitor", "fate A", "fate B", "transient", "housekeeping"], n_genes,
                         p=[0.25, 0.25, 0.25, 0.1, 0.15])
    alpha_on = np.exp(rng.normal(np.log(4.0), 0.5, n_genes))
    beta = np.exp(rng.normal(np.log(12.0), 0.3, n_genes))                # splicing rate
    gamma = np.exp(rng.normal(np.log(6.0), 0.4, n_genes))                # degradation rate
    t_on = np.where(program == "fate A", rng.uniform(branch_time, 0.75, n_genes),
                    np.where(program == "fate B", rng.uniform(branch_time, 0.75, n_genes),
                             np.where(program == "transient", rng.uniform(0.1, 0.6, n_genes), 0.0)))
    t_off = np.where(program == "progenitor", rng.uniform(branch_time - 0.1, branch_time + 0.2, n_genes),
                     np.where(program == "transient", t_on + rng.uniform(0.15, 0.3, n_genes), 2.0))
    us = np.zeros((2, len(grid), n_genes, 2))                            # lineage, time, gene, (u, s)
    for lineage, own in enumerate(["fate A", "fate B"]):
        on = (grid[:, None] >= t_on) & (grid[:, None] < t_off)
        silent = np.isin(program, ["fate A", "fate B"]) & (program != own)
        alpha = np.where(on & ~silent, alpha_on, 0.0)
        if priming > 0:                                                  # weak early bias towards own fate
            early = (program == own) & (grid[:, None] < t_on) & (grid[:, None] > 0.1)
            alpha = np.where(early, priming * alpha_on, alpha)
        alpha = np.where(program == "housekeeping", alpha_on, alpha)
        u, s = alpha[0] / beta, alpha[0] / gamma                         # start at steady state
        for k in range(len(grid)):
            us[lineage, k, :, 0], us[lineage, k, :, 1] = u, s
            u_next = alpha[k] / beta + (u - alpha[k] / beta) * np.exp(-beta * dt)   # exact for constant α
            s = s + dt * (beta * u - gamma * s)
            u = u_next
    day = rng.integers(0, n_days, n_cells)
    centre = 0.05 + 0.9 * day / max(n_days - 1, 1)
    time_ = np.clip(rng.normal(centre, 0.09), 0.0, 1.0)
    fate = rng.integers(0, 2, n_cells)                                   # eventual fate of every cell
    k = np.clip(np.round(time_ / dt).astype(int), 0, len(grid) - 1)
    u, s = us[fate, k, :, 0], us[fate, k, :, 1]
    lib = depth * np.exp(rng.normal(0, 0.25, n_cells))[:, None]
    unspliced = rng.poisson(lib * u).astype(np.int32)
    spliced = rng.poisson(lib * s).astype(np.int32)
    state = np.where(time_ < branch_time, "progenitor", np.where(fate == 0, "fate A", "fate B"))
    obs = pd.DataFrame({"time": time_, "day": day, "state": state, "fate": np.where(fate == 0, "A", "B")},
                       index=[f"cell{i:05d}" for i in range(n_cells)])
    genes = np.array([f"Gene{j:03d}" for j in range(1, n_genes + 1)], dtype=object)
    var = pd.DataFrame({"program": program, "alpha_on": alpha_on, "beta": beta, "gamma": gamma,
                        "t_on": t_on, "t_off": np.minimum(t_off, 1.0)}, index=genes)
    return dict(spliced=spliced, unspliced=unspliced, obs=obs, var=var, velocity=beta * u - gamma * s)


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_differentiation', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · Snapshots of a process

> 📍 **Where we are.** [Module 30](30_single_cell_rnaseq.ipynb) treated cells as members of discrete types. Here the
> cells are *on their way* somewhere. Before any method, we need to know what a measurement contains — and what it
> cannot contain.

### 🔬 Assay box — spliced and unspliced reads

- **What is measured.** A gene is transcribed into **pre-mRNA**, which still contains introns. The spliceosome cuts the
  introns out, giving **mature (spliced) mRNA**, which is exported, translated and eventually degraded. Standard
  scRNA-seq libraries capture both: reads that fall in introns (or span exon–intron boundaries) come from **unspliced**
  molecules, reads that span exon–exon junctions from **spliced** ones.
- **How.** No special protocol is needed: tools such as velocyto, STARsolo or kallisto | bustools re-count the reads of
  an ordinary droplet experiment into two matrices, *spliced* and *unspliced*, with the same cells and genes. In
  3′-end droplet protocols many intronic reads are thought to come from the poly-T primer binding A-rich stretches
  inside introns (internal priming) — a side effect rather than a design feature.
- **Typical sizes.** The same cells and genes as the ordinary count matrix; unspliced counts are a minority of all
  counts and therefore sparser and noisier than spliced counts.
- **Typical artefacts.** Ambiguous reads (in regions shared by several transcripts or near exon boundaries),
  genes with few or very short introns (little unspliced signal), intron retention that is regulated rather than
  transient, and nuclear vs cytoplasmic capture differences between protocols.

### What a snapshot contains

The simulator follows each gene through time with the kinetic equations of Section 4 and then samples cells on four
**sampling days**. Within a day, cells are at different **latent times** (development is not synchronised), so days
overlap. A cell's **fate** (A or B) is decided at the branch point, but the simulator records the fate every cell *will*
take — including progenitors that have not yet chosen. That is the ground truth no real experiment has.

In [ ]:
# ── 1.1 The SIMULATED time course: 2,000 cells × 200 genes, sampled on 4 days ──
diff = simulate_differentiation(np.random.default_rng(SEED + 1), n_cells=2000, n_genes=200, n_days=4)
S, U, obs, var = diff["spliced"], diff["unspliced"], diff["obs"], diff["var"]
true_vel = diff["velocity"]                       # hidden truth: ds/dt of every gene in every cell
t_true, day = obs.time.to_numpy(), obs.day.to_numpy()
fate, state = obs.fate.to_numpy(str), obs.state.to_numpy(str)
program = var.program.to_numpy(str)
print(f"SIMULATED spliced and unspliced matrices: {S.shape[0]} cells × {S.shape[1]} genes each")
print(f"counts per cell: spliced median {np.median(S.sum(1)):.0f}, unspliced median {np.median(U.sum(1)):.0f} "
      f"→ unspliced share of all counts {U.sum() / (U.sum() + S.sum()):.0%}")
print(f"zeros: spliced {np.mean(S == 0):.0%}, unspliced {np.mean(U == 0):.0%}")
print("gene programmes:", var.program.value_counts().to_dict())
print("\ncells per sampling day and true state (hidden):\n" + pd.crosstab(obs.day, obs.state).to_string())
per_day = obs.groupby("day").time.agg(["mean", "std", "min", "max"]).round(2)
print("\ntrue latent time per day (mean, sd, min, max):\n" + per_day.to_string())

🔍 **What just happened?** Each cell has two count vectors: spliced (about 1,500 counts per cell) and unspliced (about
700). In this simulation about a third of the counts are unspliced — more than in typical real data — and roughly half
of all entries are zero. Day 0 contains only progenitors; day 1 is a mixture of progenitors and cells that have just
branched; days 2 and 3 contain only committed cells. The latent times of neighbouring days overlap (the standard
deviation within a day is about 0.1, and day 1 spans true times from 0.08 to 0.68), exactly as in real time courses,
where development runs at different speeds in different cells. The genes fall into five programmes: progenitor genes
(switched off as cells branch), fate-A and fate-B genes (switched on after the branch, only in their lineage),
transient genes (on and then off) and housekeeping genes (constant).

In [ ]:
# ── 1.2 PCA of the log-normalised spliced counts, coloured by what we know and what we don't ──
X = log_normalize(S)                                    # cells × genes, log(1 + 10^4 · s / total)
pc_scores, pc_loadings, pc_evr = pca(X, n_components=30)
Z10, E2 = pc_scores[:, :10], pc_scores[:, :2]           # 10 PCs for computing, 2 PCs for drawing
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
plot_embedding(E2, t_true, ax=axes[0], s=3, title="true latent time (hidden)")
plot_embedding(E2, np.array([f"day {d}" for d in day]), ax=axes[1], s=3, title="sampling day (observed)")
plot_embedding(E2, state, ax=axes[2], s=3, title="true state (hidden)")
for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.suptitle("SIMULATED differentiation: PC1 vs PC2", y=1.02)
plt.tight_layout(); plt.show()
print(f"variance explained by PCs 1–5: {pc_evr[:5].round(3).tolist()}")

🔍 **What just happened?** The cells form a "Y": progenitors at one end, two arms for the two fates. PC1 follows time
and PC2 separates the fates — together they explain about three quarters of the variance, because this simulated
process is the only biology in the data. The sampling days (middle) are ordered along the Y but overlap. Note what is
*not* in the picture: there are no arrows. A snapshot does not say whether cells move from the progenitor end to the
arms or the other way round, nor which progenitor will choose which arm. The rest of the notebook tries to infer
order, direction and fate.

✏️ **Your turn.** Re-simulate with `n_days=8` (and the same seed). How does the overlap between consecutive days
change, and why would a finer time course help the optimal-transport method of Section 6?

---
## 2 · Pseudotime with diffusion maps

> 📍 **Where we are.** If differentiation is gradual, each cell is similar to cells slightly before and after it. A
> **random walk** that hops between similar cells will drift along the process. Measuring how far the walk has to go
> from a starting cell gives a **pseudotime**.

> 🔗 **Builds on** [Module 19](19_hmm_inference.ipynb) and [Module 20](20_hmm_learning.ipynb): the hidden chain of an
> HMM is a **Markov chain** with a row-stochastic transition matrix $a_{ij} = P(\text{to } j \mid \text{from } i)$.
> Here the *states* are the cells themselves, and the transition probabilities come from similarities.

### From a k-NN graph to a random walk

Connect each cell to its $k$ nearest neighbours in PC space and give each edge a weight

$$K_{ij} = \exp\!\Big(-\frac{\lVert z_i - z_j\rVert^2}{\sigma_i\,\sigma_j}\Big), \qquad T = D^{-1}K, \quad D = \operatorname{diag}\Big(\sum_j K_{ij}\Big).$$

*In words:* nearby cells get weights close to 1, distant ones close to 0; $\sigma_i$ is a local length scale (the
distance from cell $i$ to its $k$-th neighbour), so dense and sparse regions are treated alike. Dividing each row by its
sum turns the weights into probabilities: $T_{ij}$ is the probability that a walker on cell $i$ hops to cell $j$ in one
step. $T^t$ gives the probabilities after $t$ steps.

🧮 **Math refresher — eigenvectors of a transition matrix.** $T$ is row-stochastic, so $T\mathbf 1 = \mathbf 1$: the
constant vector is an eigenvector with eigenvalue $\lambda_0 = 1$. The other eigenvectors $\psi_1, \psi_2, \dots$ have
eigenvalues $1 > \lambda_1 \ge \lambda_2 \ge \dots$, and

$$T^t\,\psi_k = \lambda_k^t\,\psi_k .$$

*In words:* each eigenvector is a pattern over cells that the random walk does not reshuffle — it only shrinks it by
$\lambda_k$ per step. Patterns with $\lambda_k$ close to 1 decay slowly: they vary smoothly along the graph and describe
its large-scale shape (the long axis of the process, then the branches). These are the **diffusion components**
(Coifman & Lafon 2006). $T$ is not symmetric, but $D^{-1/2}KD^{-1/2}$ is, has the same eigenvalues, and its
eigenvectors $\phi_k$ give $\psi_k = D^{-1/2}\phi_k$ — so we can use a fast symmetric eigensolver (as for PCA in
[Module 14](14_pca_dimensionality_reduction.ipynb)).

> 🔗 **Builds on** [Module 14 · PCA](14_pca_dimensionality_reduction.ipynb): PCA keeps the eigenvectors of the
> covariance matrix with the largest eigenvalues (directions of largest variance); a diffusion map keeps the
> eigenvectors of the random-walk matrix with the largest eigenvalues (patterns that survive diffusion). Unlike PCA,
> the diffusion map follows curved paths, because it only trusts *local* distances.

In [ ]:
# ── 2.1 The diffusion operator and its leading eigenvectors (diffusion components) ──
def diffusion_kernel(Zc, k=15):
    """Symmetric Gaussian-kernel k-NN affinity matrix K (sparse) with adaptive widths, and its row sums."""
    n = len(Zc)
    dist, nbr = NearestNeighbors(n_neighbors=k + 1).fit(Zc).kneighbors(Zc)   # column 0 is the cell itself
    sigma = dist[:, -1]                                                      # distance to the k-th neighbour
    rows, cols, d = np.repeat(np.arange(n), k), nbr[:, 1:].ravel(), dist[:, 1:].ravel()
    K = sparse.csr_matrix((np.exp(-d ** 2 / (sigma[rows] * sigma[cols])), (rows, cols)), shape=(n, n))
    K = K.maximum(K.T)                                                       # undirected graph
    return K, np.asarray(K.sum(axis=1)).ravel()


K_aff, deg = diffusion_kernel(Z10, k=15)
print("connected components of the graph:", connected_components(K_aff)[0])
D_half = sparse.diags(1 / np.sqrt(deg))
lam, phi = eigsh(D_half @ K_aff @ D_half, k=10, which="LA", v0=np.random.default_rng(SEED).random(len(deg)))
order = np.argsort(lam)[::-1]
lam, phi = lam[order], phi[:, order]
psi = phi / np.sqrt(deg)[:, None]                       # right eigenvectors of T = D⁻¹K
psi /= np.linalg.norm(psi, axis=0)
psi[:, 1] *= np.sign(stats.spearmanr(psi[:, 1], day)[0])  # eigenvector signs are arbitrary: let DC1 grow with day
print("eigenvalues λ0…λ5:", lam[:6].round(5).tolist())
print(f"check: ψ0 is constant (sd/mean = {psi[:, 0].std() / abs(psi[:, 0].mean()):.1e}); "
      f"T ψ1 = λ1 ψ1 holds to {np.abs((K_aff @ psi[:, 1]) / deg - lam[1] * psi[:, 1]).max():.1e}")

🔍 **What just happened?** The graph is connected (one component), so only one eigenvalue equals 1 and its eigenvector
$\psi_0$ is constant, as the refresher predicted; the check $T\psi_1 = \lambda_1\psi_1$ holds to numerical precision.
The next eigenvalues are all *very* close to 1 (from 0.9999 down to 0.998): a walker that hops between 15 neighbours moves only a tiny
distance per step, so large-scale patterns decay slowly. What matters is their order and how close each is to 1.

### Diffusion pseudotime (DPT)

A natural distance between two cells is how differently random walks started from them spread out. Summing that
difference over *all* numbers of steps $t = 1, 2, \dots$ (Haghverdi *et al.* 2016) gives, in the eigenvector basis,

$$\operatorname{dpt}(x, y)^2 = \sum_{k \ge 1}\Big(\frac{\lambda_k}{1-\lambda_k}\Big)^2\big(\psi_k(x) - \psi_k(y)\big)^2,
\qquad \text{pseudotime}(x) = \operatorname{dpt}(\text{root}, x).$$

*In words:* each diffusion component contributes the difference between the two cells, weighted by
$\lambda_k + \lambda_k^2 + \lambda_k^3 + \dots = \lambda_k/(1-\lambda_k)$ — slowly decaying (large-scale) components
count much more than noisy, fast-decaying ones. The pseudotime of a cell is its diffusion distance from a chosen
**root** cell.

⚠️ **Why pseudotime needs a root.** Distances are symmetric: the graph alone cannot tell the start of a process from its
end. The direction must come from outside: known biology (stem-cell markers), the sampling day, or RNA velocity
(Section 4). We use the experiment: the root is the day-0 cell at the early extreme of DC1.

In [ ]:
# ── 2.2 Diffusion pseudotime from a root cell, and 🧪 its agreement with the true time ──
weights = lam[1:] / (1 - lam[1:])
M_dc = psi[:, 1:] * weights                              # weighted diffusion coordinates
day0 = np.flatnonzero(day == 0)
root = day0[np.argmin(psi[day0, 1])]                     # the most 'upstream' day-0 cell on DC1
dpt = np.linalg.norm(M_dc - M_dc[root], axis=1)
dpt /= dpt.max()

euclid = np.linalg.norm(Z10 - Z10[root], axis=1)         # straight-line distance in PC space
candidates = {"PC1 (oriented by day)": pc_scores[:, 0] * np.sign(stats.spearmanr(pc_scores[:, 0], day)[0]),
              "Euclidean distance from root": euclid, "diffusion pseudotime": dpt}
rows = {}
for name, pt in candidates.items():
    rows[name] = {"all cells": stats.spearmanr(pt, t_true)[0],
                  "fate A branch": stats.spearmanr(pt[fate == "A"], t_true[fate == "A"])[0],
                  "fate B branch": stats.spearmanr(pt[fate == "B"], t_true[fate == "B"])[0],
                  "within a day (mean)": np.mean([stats.spearmanr(pt[day == d], t_true[day == d])[0] for d in range(4)])}
print(f"root cell: true time {t_true[root]:.2f}, state {state[root]}")
print("🧪 Spearman correlation with the true latent time:")
print(pd.DataFrame(rows).T.round(3).to_string())
print(f"(the sampling day alone: {stats.spearmanr(day, t_true)[0]:.3f})")

In [ ]:
# ── 2.3 Diffusion components and pseudotime ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
plot_embedding(psi[:, 1:3], t_true, ax=axes[0], s=3, title="DC1 vs DC2, true time (hidden)")
plot_embedding(psi[:, 1:3], state, ax=axes[1], s=3, title="DC1 vs DC2, true state (hidden)")
for ax in axes[:2]:
    ax.scatter(*psi[root, 1:3], s=120, marker="*", color="k", zorder=5)
    ax.set_xlabel("DC1"); ax.set_ylabel("DC2")
for f, c in [("A", CLASS_COLORS[0]), ("B", CLASS_COLORS[4])]:
    m = fate == f
    axes[2].scatter(t_true[m], dpt[m], s=3, color=c, alpha=0.6, label=f"fate {f}")
axes[2].set_xlabel("true latent time (hidden)"); axes[2].set_ylabel("diffusion pseudotime")
axes[2].set_title("pseudotime recovers the order of cells"); axes[2].legend(markerscale=4)
plt.suptitle("SIMULATED differentiation: diffusion map (★ = root cell)", y=1.02)
plt.tight_layout(); plt.show()
late = t_true > 0.6
print("🧪 AUROC of each diffusion component for the fate of late cells (time > 0.6):",
      {f"DC{j}": round(max(a := roc_auc_score(fate[late] == "A", psi[late, j]), 1 - a), 3) for j in range(1, 5)})

🔍 **What just happened?** Diffusion pseudotime orders the cells almost perfectly (Spearman ρ = 0.98 with the hidden
latent time, and 0.99 within each branch). PC1 does worse (0.94): it is a straight axis, and the two arms of the "Y" bend
away from it. The straight-line distance from the root does almost as well as DPT here, because in 10 PCs our simulated
arms are nearly straight; on curved or looping real trajectories the gap is usually larger. The most demanding column
is *within a day*: the sampling day alone already gives a coarse ordering (ρ = 0.96), so the real test is whether a
method can order cells collected on the *same* day. DPT does (mean within-day ρ = 0.77), PC1 much less (0.59). The
scatter plot shows the remaining weakness: the earliest cells ($t < 0.15$) get pseudotime close to 0 (two small groups,
near 0 and near 0.12), because almost nothing changes in the simulated progenitors before the first genes switch; then
pseudotime jumps to about 0.33 as soon as they do. Pseudotime can only order cells whose expression differs, and it is
an *ordering*, not a clock: its spacing reflects how much expression changes, not how much time passes. The diffusion components divide the work: DC1 is time, DC2 is the branch (it classifies the fate
of late cells perfectly, AUROC 1.0). Pseudotime alone is one number per cell and does not say which branch a cell is
on — that is the job of the trajectory skeleton (Section 3).

✏️ **Your turn.** Put the root at the tip of a branch instead: `root = np.flatnonzero(day == 3)[np.argmax(psi[day == 3, 2])]`
and re-run 2.2. What happens to the correlation with true time on each branch? This is what an unlucky root choice
does to a real analysis.

### 📄 Research connection — Pseudotime and trajectory inference

- **What the field does.** Monocle (Trapnell *et al.* 2014) ordered cells along a minimum spanning tree and named the result
    "pseudotime". Diffusion maps (Coifman & Lafon 2006) entered single-cell biology with diffusion pseudotime (DPT,
    Haghverdi *et al.* 2016), and PAGA (Wolf *et al.* 2019) summarised the k-NN graph as a graph of clusters whose edges
    reflect connectivity — a skeleton that can contain trees, cycles and disconnected pieces. Slingshot fits principal
    curves through a cluster tree; Palantir (Setty *et al.* 2019) models differentiation as a Markov chain and reports
    fate probabilities. Saelens *et al.* (2019) benchmarked dozens of trajectory-inference methods and found that the
    best choice depends on the topology of the process (linear, bifurcating, tree, cycle).
- **What this notebook builds.** A Gaussian-kernel k-NN random walk, its diffusion components and DPT from scratch, checked against the true
    latent time; in Section 3 a cluster tree and a PAGA-like connectivity graph. Scanpy's `sc.tl.diffmap`, `sc.tl.dpt`
    and `sc.tl.paga` implement the production versions.
- **Where the frontier is.** Pseudotime assumes that cells sample a continuous process densely and that similarity means temporal
    proximity — assumptions that fail for fast transitions, cycles (the cell cycle) and convergent paths in which two
    origins produce similar cells. Current work combines pseudotime with real time (sampling days, metabolic labelling),
    lineage barcodes and velocity, and asks how uncertain a trajectory is rather than drawing a single tree.
- **Read next.** Haghverdi *et al.* 2016 (*Nature Methods*), *Diffusion pseudotime robustly reconstructs lineage branching*; Saelens *et al.* 2019 (*Nature Biotechnology*), *A comparison of single-cell trajectory inference methods*.

---
## 3 · A trajectory skeleton

> 📍 **Where we are.** Pseudotime is one number per cell. To see the *shape* of the process — where it branches and
> into how many arms — we summarise the cells as a small graph of clusters.

The recipe of Monocle 1, TSCAN and Slingshot: cluster the cells, then connect the cluster centres with a **minimum
spanning tree (MST)** — the set of edges that connects all centres with the smallest total length and no cycles. A
second recipe, in the spirit of **PAGA**: two clusters are connected if many k-NN edges run between them, compared with
what random wiring would give,

$$c_{ab} = \frac{E_{ab}}{e_a\,e_b / (2m)} ,$$

*In words:* $E_{ab}$ is the number of k-NN edges between clusters $a$ and $b$, $e_a$ the number of edge ends in cluster
$a$ and $m$ the total number of edges; the denominator is the number of edges expected if edges were placed at random
with the same cluster sizes (the null model of modularity in [Module 30](30_single_cell_rnaseq.ipynb)). A tree over
$-\log c_{ab}$ keeps the most strongly connected pairs.

> 🔗 **Builds on** [Module 12 · Clustering](12_clustering_kmeans_hierarchical.ipynb) (k-means) and
> [Module 17](17_bayesian_networks_inference_learning.ipynb) (the Chow–Liu algorithm also learns a tree structure as a
> maximum spanning tree over pairwise scores).

In [ ]:
# ── 3.1 Clusters, a centroid MST and a PAGA-like connectivity tree ──
N_CL = 10
cl = KMeans(N_CL, n_init=10, random_state=SEED).fit_predict(Z10)
cent = np.array([Z10[cl == c].mean(axis=0) for c in range(N_CL)])
mst_cent = minimum_spanning_tree(np.linalg.norm(cent[:, None] - cent[None], axis=2)).toarray()

nbr15 = NearestNeighbors(n_neighbors=16).fit(Z10).kneighbors(Z10)[1][:, 1:]
E_ab = np.zeros((N_CL, N_CL))
np.add.at(E_ab, (np.repeat(cl, 15), cl[nbr15.ravel()]), 1)
E_ab = E_ab + E_ab.T                                     # undirected edge counts
e_a = E_ab.sum(axis=1)
conn = E_ab / np.outer(e_a, e_a) * e_a.sum()             # observed / expected
np.fill_diagonal(conn, 0)
cost = np.where(conn > 0, -np.log(np.maximum(conn, 1e-12)) + 10, 0)   # +10 keeps every existing edge positive
mst_paga = minimum_spanning_tree(cost).toarray()

cl_time = np.array([t_true[cl == c].mean() for c in range(N_CL)])        # hidden truth, for grading only
cl_state = np.array([pd.Series(state[cl == c]).mode()[0] for c in range(N_CL)])
comp = pd.crosstab(pd.Series(cl, name="cluster"), pd.Series(state, name="state"), normalize="index").round(2)
comp.insert(0, "mean true time", cl_time.round(2)); comp.insert(0, "cells", np.bincount(cl))
comp["majority"] = cl_state
print("clusters with their hidden composition (fractions of cells in each true state):")
print(comp.sort_values("mean true time").to_string())

In [ ]:
# ── 3.2 🧪 Do the trees agree with the true lineage (progenitor → A, progenitor → B)? ──
def grade_tree(mst, root_cluster):
    """Edges between the two fates, edges going back in true time when the tree is rooted, and the leaves."""
    adj = ((mst + mst.T) > 0).astype(int)
    edges = np.argwhere(np.triu(adj))
    cross = sum({cl_state[a], cl_state[b]} == {"fate A", "fate B"} for a, b in edges)
    order_bfs, pred = breadth_first_order(adj, root_cluster, directed=False)
    backwards = sum(cl_time[c] < cl_time[pred[c]] for c in order_bfs[1:])
    leaves = [c for c in range(N_CL) if adj[c].sum() == 1 and c != root_cluster]
    return edges, cross, backwards, leaves


root_cl = cl[root]
for name, mst in [("centroid MST", mst_cent), ("PAGA-like tree", mst_paga)]:
    edges, cross, backwards, leaves = grade_tree(mst, root_cl)
    hubs = [int(c) for c in range(N_CL) if ((mst + mst.T) > 0)[c].sum() >= 3]
    print(f"🧪 {name}: {len(edges)} edges; edges joining an A-majority to a B-majority cluster: {cross}; "
          f"edges going back in true time: {backwards}")
    leaf_info = [(int(c), str(cl_state[c]), round(float(cl_time[c]), 2)) for c in leaves]
    print(f"     branch-point cluster(s) (≥ 3 edges): {hubs}; leaves (cluster, majority, mean time): {leaf_info}")

In [ ]:
# ── 3.3 The skeleton drawn on the PCA map ──
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cent2 = np.array([E2[cl == c].mean(axis=0) for c in range(N_CL)])
for ax, mst, title in [(axes[0], mst_cent, "minimum spanning tree of cluster centres"),
                       (axes[1], mst_paga, "PAGA-like tree (edge width = connectivity)")]:
    plot_embedding(E2, state, ax=ax, s=2, legend=(ax is axes[0]))
    for a, b in np.argwhere(np.triu((mst + mst.T) > 0)):
        lw = 1 + 2 * min(conn[a, b], 3) if mst is mst_paga else 2
        ax.plot(*cent2[[a, b]].T, color="k", lw=lw, zorder=4)
    ax.scatter(*cent2.T, s=60, color="white", edgecolor="k", zorder=5)
    for c in range(N_CL):
        ax.text(*cent2[c], str(c), fontsize=7, ha="center", va="center", zorder=6)
    ax.scatter(*cent2[root_cl], s=160, facecolor="none", edgecolor="k", lw=2, zorder=6)
    ax.set_title(title); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.suptitle("SIMULATED: trajectory skeletons over 10 k-means clusters (circled = root cluster)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Ten k-means clusters chop the "Y" into segments: two pure progenitor clusters, a cluster at
the branch point, and three or four clusters along each arm. Both trees reproduce the true lineage: rooted at the root
cell's cluster, no edge goes back in true time; there is exactly one branching cluster, and the two leaves are the
latest clusters of the two arms. The one edge reported as joining "A" to "B" deserves a look at the composition table:
it attaches the B arm to cluster 7, which is 51 % fate A and 49 % fate B — the cells that have *just* branched, in both
directions. Labelling a cluster by its majority hides that it *is* the branch point; that is a limitation of
summarising a continuum by clusters, not a wrong edge. On this clean simulation the two constructions agree; they
differ on real data, where cluster centres can be close in space but poorly connected through cells (the centroid MST
will connect them anyway), and where processes contain cycles or disconnected parts that a *tree* cannot represent but
a PAGA *graph* can.

⚠️ **A tree is always found.** An MST connects *any* set of clusters into a tree, even clusters of unrelated cell types.
The existence of a tree is not evidence of a lineage; only connectivity through intermediate cells (and, ideally,
independent evidence such as time points or lineage barcodes) supports a path.

✏️ **Your turn.** Change `N_CL` to 4 and to 20 and re-run 3.1–3.3. Which features of the tree are stable, and when does
the MST start to produce small side branches that are not real lineages?

---
## 4 · RNA velocity

> 📍 **Where we are.** Pseudotime needed a root because a snapshot has no arrow. RNA velocity extracts an arrow from the
> snapshot itself, using the two kinds of reads of Section 1. First the physics of a single gene.

### The kinetic model

For one gene in one cell, with $u$ the number of unspliced and $s$ the number of spliced molecules:

$$\frac{du}{dt} = \alpha(t) - \beta\,u, \qquad \frac{ds}{dt} = \beta\,u - \gamma\,s .$$

*In words:* unspliced molecules are made at the **transcription rate** $\alpha$ (switched on or off by regulation) and
turned into spliced molecules at the **splicing rate** $\beta$ per molecule; spliced molecules appear by splicing and
disappear by **degradation** at rate $\gamma$ per molecule. The **RNA velocity** of the gene is $ds/dt$: positive when
the gene's mRNA is accumulating, negative when it is decaying.

🧮 **Math refresher — differential equations and steady states.** An ordinary differential equation (ODE) gives the
*rate of change* of a quantity as a function of its current value. [Module 04](04_calculus_and_optimization.ipynb)
taught that a derivative is a slope; here we know the slope and want the curve. The simplest numerical solution is
**Euler's method**: take a small time step $\Delta t$ and update $u \leftarrow u + \Delta t\,(\alpha - \beta u)$, which
is gradient descent's update with "slope" in place of "gradient". A **steady state** is where nothing changes: setting
both derivatives to zero gives

$$u^\ast = \frac{\alpha}{\beta}, \qquad s^\ast = \frac{\alpha}{\gamma} \quad\Rightarrow\quad u^\ast = \frac{\gamma}{\beta}\,s^\ast .$$

*In words:* if transcription stays constant long enough, both kinds of molecules settle at levels proportional to
$\alpha$, and — whatever $\alpha$ is — the steady-state points lie on one straight line through the origin with slope
$\gamma/\beta$. A cell **above** the line has more unspliced RNA than steady state predicts: the gene was recently
switched on and $s$ will rise ($ds/dt = \beta(u - \tfrac{\gamma}{\beta}s) > 0$). A cell **below** the line is
switching off.

In [ ]:
# ── 4.1 One gene, switched on and then off: solving the kinetic equations step by step ──
def solve_kinetics(alpha, beta, gamma, dt):
    """Euler integration of du/dt = alpha(t) − beta·u, ds/dt = beta·u − gamma·s from u = s = 0.
    `alpha` holds the transcription rate at each time step; returns the arrays u(t) and s(t)."""
    u, s = np.zeros(len(alpha)), np.zeros(len(alpha))
    for k in range(1, len(alpha)):
        u[k] = u[k - 1] + dt * (alpha[k - 1] - beta * u[k - 1])    # new = old + time step × slope
        s[k] = s[k - 1] + dt * (beta * u[k - 1] - gamma * s[k - 1])
    return u, s


grid = np.linspace(0, 1, 2001); dt = grid[1] - grid[0]
alpha_t = np.where((grid >= 0.1) & (grid < 0.7), 4.0, 0.0)       # switched on at t = 0.1, off at t = 0.7
u_t, s_t = solve_kinetics(alpha_t, beta=12.0, gamma=6.0, dt=dt)
k_off = np.searchsorted(grid, 0.7)
print(f"just before switching off: u = {u_t[k_off - 1]:.3f} (u* = α/β = {4 / 12:.3f}), "
      f"s = {s_t[k_off - 1]:.3f} (s* = α/γ = {4 / 6:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(grid, u_t, color=CLASS_COLORS[1], label="unspliced u")
axes[0].plot(grid, s_t, color=CLASS_COLORS[0], label="spliced s")
axes[0].fill_between(grid, 0, 0.05 * (alpha_t > 0), color="0.7", label="transcription on")
axes[0].set_xlabel("time"); axes[0].set_ylabel("molecules (per unit depth)")
axes[0].legend(); axes[0].set_title("u rises first, s follows")
sc = axes[1].scatter(s_t[::10], u_t[::10], c=grid[::10], s=8, cmap="viridis")
axes[1].plot([0, 0.75], [0, 0.75 * 6 / 12], "k--", lw=1, label="steady states: u = (γ/β) s")
for k in (260, 450, 1420, 1600):                                   # arrows show the direction of travel
    axes[1].annotate("", xy=(s_t[k + 40], u_t[k + 40]), xytext=(s_t[k], u_t[k]), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[1].set_xlabel("spliced s"); axes[1].set_ylabel("unspliced u"); axes[1].legend(loc="upper left")
axes[1].set_title("phase portrait: above the line = inducing, below = repressing"); plt.colorbar(sc, ax=axes[1], label="time")
plt.tight_layout(); plt.show()

🔍 **What just happened?** When transcription switches on, unspliced RNA rises first (it only needs transcription) and
spliced RNA follows with a delay (it needs splicing); both approach the steady state $u^\ast = 1/3$, $s^\ast = 2/3$
(when transcription stops at $t = 0.7$, the solver has $u = 0.333$ and $s = 0.631$ — $s$, which turns over more
slowly, is still creeping towards 0.667). When transcription stops, $u$ collapses quickly ($\beta = 12$ is fast) while
$s$ decays more slowly. In the **phase portrait** ($u$ against $s$) one cycle traces a loop: the upper arc (induction) lies
above the steady-state line, the lower arc (repression) below it. A snapshot of many cells at different moments of this
cycle scatters points along the loop — and the *side of the line* each cell is on tells its direction. That is the whole
idea of RNA velocity.

### Phase portraits of the simulated genes

Real counts are noisy, so velocity methods first normalise $u$ and $s$ by a size factor and then average each cell
with its $k$ nearest neighbours in PC space (scVelo calls these averages the first-order **moments**). The averaging
trades resolution for noise reduction, as k-NN smoothing always does.

In [ ]:
# ── 4.2 Size factors and k-NN-smoothed spliced/unspliced counts ("moments") ──
size = S.sum(axis=1) + U.sum(axis=1)
size = size / np.median(size)                                     # one size factor per cell, shared by u and s
nbr30 = NearestNeighbors(n_neighbors=30).fit(Z10).kneighbors(Z10)[1]   # column 0 is the cell itself
Ms = (S / size[:, None])[nbr30].mean(axis=1)                      # cells × genes
Mu = (U / size[:, None])[nbr30].mean(axis=1)
gamma_true = (var.gamma / var.beta).to_numpy()                    # hidden truth: slope of the steady-state line


def fit_steady_state(Mu, Ms, perc=5):
    """Steady-state velocity model: slope γ′ of u on s through the origin, fitted on the cells in the lowest and highest
    `perc` % of (u/max u + s/max s) — the cells most likely to sit at the 'off' and 'fully on' steady states.
    Also returns R² of the line over all cells (how well the gene fits the steady-state picture)."""
    G = Ms.shape[1]
    slope, r2 = np.zeros(G), np.zeros(G)
    for g in range(G):
        u, s = Mu[:, g], Ms[:, g]
        w = u / u.max() + s / s.max()
        lo, hi = np.percentile(w, [perc, 100 - perc])
        ext = (w <= lo) | (w >= hi)
        slope[g] = (u[ext] @ s[ext]) / (s[ext] @ s[ext])          # least squares through the origin (Module 06)
        r2[g] = 1 - np.sum((u - slope[g] * s) ** 2) / np.sum((u - u.mean()) ** 2)
    return slope, r2


gamma_hat, r2_fit = fit_steady_state(Mu, Ms)
print(f"moments: {Ms.shape[0]} cells × {Ms.shape[1]} genes, each the average of 30 neighbours; "
      f"γ′ fitted for all {len(gamma_hat)} genes (median γ′ = {np.median(gamma_hat):.2f}, "
      f"median true γ/β = {np.median(gamma_true):.2f})")

In [ ]:
# ── 4.3 Phase portraits of three genes: progenitor (switching off), fate A (switching on), transient ──
picks = {p: np.flatnonzero(program == p)[0] for p in ["progenitor", "fate A", "transient"]}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (p, g) in zip(axes, picks.items()):
    sc = ax.scatter(Ms[:, g], Mu[:, g], c=t_true, s=3, cmap="viridis")
    smax = Ms[:, g].max()
    ax.plot([0, smax], [0, gamma_true[g] * smax], "k--", lw=1.2, label="true γ/β")
    ax.plot([0, smax], [0, gamma_hat[g] * smax], color=CLASS_COLORS[1], lw=1.5, label="fitted γ′")
    ax.set_title(f"{var.index[g]} ({p}, switches {'off' if p == 'progenitor' else 'on'} at "
                 f"t = {var.t_off.iloc[g] if p == 'progenitor' else var.t_on.iloc[g]:.2f})")
    ax.set_xlabel("smoothed spliced"); ax.set_ylabel("smoothed unspliced"); ax.legend(fontsize=7, loc="upper left")
plt.colorbar(sc, ax=axes[2], label="true time (hidden)")
plt.suptitle("SIMULATED phase portraits (colour = hidden true time)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each panel is the loop of 4.1 seen through noisy cells. The progenitor gene starts at its "on"
steady state (top right, early cells), and as it switches off the cells drop *below* the line towards the origin. The
fate-A gene starts at zero; after it switches on, the cells of lineage A rise *above* the line (inducing) and approach
the upper steady state, while lineage-B cells stay at the origin. The transient gene goes round the whole loop. The
fitted slope (red) is close to the true $\gamma/\beta$ (dashed) for the progenitor gene, but for genes that are still
inducing at the end of the experiment it is too steep — we return to that in Section 5.

In [ ]:
# ── 4.4 🧪 Ground truth for γ′ and for the velocities ──
V = Mu - gamma_hat * Ms                                           # steady-state velocity (up to the factor β)
r_gene = np.array([np.corrcoef(V[:, g], true_vel[:, g])[0, 1] if true_vel[:, g].std() > 0 else np.nan
                   for g in range(V.shape[1])])
dynamic = np.abs(true_vel).max(axis=0) > 1e-6                    # housekeeping genes never change (true v = 0)
strong = (np.abs(true_vel) > 0.1 * np.abs(true_vel).max(axis=0)) & dynamic   # cells where the gene really changes
sign_ok = np.array([np.mean(np.sign(V[strong[:, g], g]) == np.sign(true_vel[strong[:, g], g])) if strong[:, g].any()
                    else np.nan for g in range(V.shape[1])])
tab = pd.DataFrame({"programme": program, "log2(γ′ / true γ/β)": np.log2(gamma_hat / gamma_true),
                    "corr(v, true velocity)": r_gene, "sign correct where |v| is large": sign_ok})
print(f"🧪 Spearman(γ′, true γ/β) over all genes = {stats.spearmanr(gamma_hat, gamma_true)[0]:.3f}")
print("🧪 medians per gene programme:")
print(tab.groupby("programme").median().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for k, p in enumerate(["progenitor", "fate A", "fate B", "transient", "housekeeping"]):
    m = program == p
    axes[0].scatter(gamma_true[m], gamma_hat[m], s=12, color=CLASS_COLORS[k], label=p)
    jitter = np.random.default_rng(k).uniform(-0.2, 0.2, m.sum())
    axes[1].scatter(np.full(m.sum(), k) + jitter, r_gene[m], s=10, color=CLASS_COLORS[k])
lims = [gamma_true.min() * 0.8, gamma_hat.max() * 1.2]
axes[0].plot(lims, lims, "k--", lw=1); axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("true γ/β (hidden)"); axes[0].set_ylabel("estimated γ′")
axes[0].legend(fontsize=7); axes[0].set_title("steady-state slopes")
axes[1].set_xticks(range(5)); axes[1].set_xticklabels(["progenitor", "fate A", "fate B", "transient", "housek."], fontsize=8)
axes[1].set_ylabel("corr(estimated, true velocity)"); axes[1].set_title("per-gene velocity accuracy (housekeeping: true v = 0)")
plt.suptitle("SIMULATED: RNA velocity against the hidden kinetics", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The estimated slopes track the true $\gamma/\beta$ (Spearman ρ = 0.88), but not uniformly:
progenitor and housekeeping genes, which start the experiment *at* steady state, are estimated almost without bias
(median log2 ratio ≈ 0), whereas fate genes are overestimated by about 0.33 log2 units (≈ 25 %) and transient genes by
about 0.7 (≈ 60 %). The per-gene velocities still correlate well with the true velocities (medians 0.84–0.93). The sign
is a harsher test: where a gene is really changing, the estimated velocity has the right sign in 100 % of cells for
progenitor genes and 95 % for transient genes, but only in about 78 % for fate genes — the price of their too-steep
slopes (Section 5). Housekeeping genes have true velocity exactly 0, so there is nothing to correlate or to get right:
their estimated velocities are pure noise, which is why velocity tools select **velocity genes** with good fits before
going further.

> 🔗 **Builds on** [Module 06 · Linear regression](06_linear_regression_gwas.ipynb): γ′ is a least-squares slope through
> the origin, $\hat\gamma' = \sum u_i s_i / \sum s_i^2$, fitted on a deliberately chosen subset of cells. Choosing that
> subset is the modelling assumption — and the source of the bias.

### From gene velocities to an arrow per cell

A velocity vector $v_i$ (one entry per gene) says where cell $i$ is heading in expression space. To turn it into a
movement on the graph, compare it with the displacement $\delta_{ij} = s_j - s_i$ towards each neighbour $j$:

$$\pi_{ij} = \frac{\exp\big(\kappa\,\cos(\delta_{ij}, v_i)\big)}{\sum_{j'}\exp\big(\kappa\,\cos(\delta_{ij'}, v_i)\big)} .$$

*In words:* a neighbour lying in the direction the velocity points (cosine near +1) gets a high transition
probability, one lying behind the cell (cosine near −1) a low one; $\kappa$ sets how sharp the preference is (scVelo uses
the same exponential form). The rows of $\pi$ form a **velocity-directed Markov chain** — unlike the diffusion of
Section 2, it has a direction. To draw arrows on a 2-D map, each cell's arrow is the $\pi$-weighted average of the unit
vectors towards its neighbours in the map, minus the unweighted average (so that a cell with no preferred direction gets
no arrow).

In [ ]:
# ── 4.5 The velocity graph and arrows on the PCA map ──
nbr = nbr30[:, 1:]                                                # 29 neighbours (drop the cell itself)
vgenes = r2_fit > 0.1                                             # 'velocity genes': the line fits reasonably


def velocity_transitions(Vel, Ms, nbr, kappa=10.0):
    """Transition probabilities from every cell to its neighbours, from the cosine between the cell's velocity and
    the displacement towards each neighbour (scVelo-style velocity graph). Returns probabilities and cosines."""
    D = Ms[nbr] - Ms[:, None, :]                                  # cells × neighbours × genes
    cos = np.einsum("ikg,ig->ik", D, Vel) / (np.linalg.norm(D, axis=2) * np.linalg.norm(Vel, axis=1)[:, None] + 1e-12)
    P = np.exp(kappa * cos)
    return P / P.sum(axis=1, keepdims=True), cos


def embed_arrows(P, E, nbr):
    """Velocity arrows in a 2-D embedding E: π-weighted minus uniform average of unit vectors to the neighbours."""
    dE = E[nbr] - E[:, None, :]
    dE /= np.linalg.norm(dE, axis=2, keepdims=True) + 1e-12
    return (P[:, :, None] * dE).sum(axis=1) - dE.mean(axis=1)


def forward_fraction(P, nbr, t):
    """Fraction of cells whose expected next step (under P) moves later in true time than a uniform random step."""
    step = (P * (t[nbr] - t[:, None])).sum(axis=1) - (t[nbr] - t[:, None]).mean(axis=1)
    return np.mean(step > 0), step


rng4 = np.random.default_rng(SEED + 4)
P_est, cos_est = velocity_transitions(V[:, vgenes], Ms[:, vgenes], nbr)
P_true, _ = velocity_transitions(true_vel[:, vgenes], Ms[:, vgenes], nbr)
P_rand, _ = velocity_transitions(rng4.normal(size=V[:, vgenes].shape), Ms[:, vgenes], nbr)   # random directions
P_shuf, _ = velocity_transitions(V[rng4.permutation(len(V))][:, vgenes], Ms[:, vgenes], nbr)  # other cells' velocities
print(f"velocity genes: {vgenes.sum()} of {len(vgenes)};  excluded by programme:",
      pd.Series(program[~vgenes]).value_counts().to_dict())

In [ ]:
# ── 4.6 🧪 Do the arrows point forward in (hidden) time? ──
rows = {}
for name, P in [("estimated velocity", P_est), ("true velocity (ceiling)", P_true),
                ("random directions (control)", P_rand), ("velocity of a random other cell (control)", P_shuf)]:
    frac, step = forward_fraction(P, nbr, t_true)
    inner = (t_true > 0.1) & (t_true < 0.9)
    rows[name] = {"all cells": frac, "cells with 0.1 < t < 0.9": np.mean(step[inner] > 0),
                  "cells with t ≤ 0.1 or ≥ 0.9": np.mean(step[~inner] > 0)}
print("🧪 fraction of cells whose velocity step goes forward in true time:")
print(pd.DataFrame(rows).T.round(3).to_string())

def grid_arrows(E, A, n=18, min_cells=5):
    """Average arrows of cells falling in the same square of an n × n grid (for a readable plot)."""
    gx = np.linspace(E[:, 0].min(), E[:, 0].max(), n + 1); gy = np.linspace(E[:, 1].min(), E[:, 1].max(), n + 1)
    ix, iy = np.clip(np.digitize(E[:, 0], gx) - 1, 0, n - 1), np.clip(np.digitize(E[:, 1], gy) - 1, 0, n - 1)
    out = [(E[m].mean(0), A[m].mean(0)) for i in range(n) for j in range(n) if (m := (ix == i) & (iy == j)).sum() >= min_cells]
    return np.array([o[0] for o in out]), np.array([o[1] for o in out])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, P, title in [(axes[0], P_est, "estimated steady-state velocity"), (axes[1], P_true, "true velocity (hidden)")]:
    plot_embedding(E2, state, ax=ax, s=2, legend=(ax is axes[0]))
    pos, arr = grid_arrows(E2, embed_arrows(P, E2, nbr))
    ax.quiver(pos[:, 0], pos[:, 1], arr[:, 0], arr[:, 1], angles="xy", scale_units="xy", scale=0.35, width=0.004, color="k")
    ax.set_title(title); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.suptitle("SIMULATED: velocity arrows projected onto PC1–PC2 (grid averages)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the estimated velocities, 86 % of cells take an expected step forward in true time —
close to the 88 % reached with the *true* velocities pushed through the same machinery. Inside the process
($0.1 < t < 0.9$) the figures are 95 % and 97 %; the misses concentrate at the ends, where the simulated time is clipped,
there are no later cells to step to and the true velocities are small. The controls need a moment's thought. Random
directions give about 50 %, as they should. But giving each cell the velocity of a *random other cell* still scores
69 %: in this process most genes move the same way in every cell (progenitor genes fall, fate genes rise), so even a
borrowed velocity points roughly "forward". The fair bar for cell-specific information is therefore about 69 %, not
50 % — and the estimated velocity clears it comfortably. The arrows on the PCA map flow from the progenitors into
both arms, as the true velocities do. For the first time we have a direction **without** using the sampling days: velocity alone
says which end of the "Y" is the start.

⚠️ **Read arrows with care.** The arrows on a 2-D map are a projection of a projection: the velocity of 200 genes is
turned into a step on a k-NN graph and then into a 2-D arrow. Different embeddings or neighbourhood sizes can change the
picture, and Gorin *et al.* (2022) caution against interpreting these plots quantitatively. Use the transition
probabilities (or the 🧪 checks) for analysis, and the arrows for looking.

✏️ **Your turn.** Change `kappa` in `velocity_transitions` to 2 and to 50 and re-run 4.5–4.6. How do the forward
fractions and the arrows change? What does a very large κ do to the Markov chain?

### 📄 Research connection — RNA velocity and its limits

- **What the field does.** La Manno *et al.* (2018) introduced RNA velocity with the steady-state model used here (velocyto): fit
    γ′ on extreme cells, compute $u - \gamma' s$, extrapolate each cell a short time ahead, and project the result onto
    an embedding with transition probabilities built from the correlation between each cell's velocity and its
    displacements to neighbours. Bergen *et al.* (2020) generalised it in **scVelo**: the *dynamical model* fits each
    gene's full switching kinetics (α, β, γ, switching time and a latent time per cell) by expectation–maximisation, and
    its cosine-similarity "velocity graph" is the construction used in 4.5.
- **What this notebook builds.** The kinetic ODEs solved numerically, the steady-state slope γ′ (graded against the true γ/β), velocities
    (graded per gene against the true ds/dt), the cosine-based velocity graph, embedded arrows and a forward-in-time
    check against hidden latent time — a check that real data can approximate only with lineage tracing or metabolic
    labelling.
- **Where the frontier is.** Bergen *et al.* (2021) list the open challenges: genes that are not at steady state, several kinetic
    regimes for one gene across lineages, transcriptional bursts and boosts, low unspliced counts, and the projection
    onto embeddings. Gorin *et al.* (2022) argue for explicit stochastic models of transcription instead of smoothed
    deterministic fits. As of 2025, active directions include metabolic labelling of new RNA (which measures time
    directly), probabilistic and deep generative velocity models that report uncertainty, and velocity for other
    modalities (chromatin, protein).
- **Read next.** Bergen *et al.* 2021 (*Molecular Systems Biology*), *RNA velocity — current challenges and future perspectives*; La Manno *et al.* 2018 (*Nature*), *RNA velocity of single cells*.

---
## 5 · When velocity fails

> 📍 **Where we are.** Velocity worked well above — but our simulator obeys the steady-state model's assumptions
> almost exactly. Here we look for the cracks: first inside our own data, then by provoking a failure that real data
> are known to contain.

**Failure 1 — genes that never reach steady state.** The steady-state fit assumes that the cells with the highest
$u + s$ are sitting *on* the steady-state line. A gene switched on late in the experiment may still be inducing in the
last cells; then the "extreme" cells lie *above* the line, and the fitted slope is too steep. A too-steep line makes
the velocity $u - \gamma' s$ too small, and negative for cells that are in fact still accumulating mRNA.

In [ ]:
# ── 5.1 Late-switching genes: the steady-state slope is biased ──
fate_genes = np.isin(program, ["fate A", "fate B"])
t_on = var.t_on.to_numpy()
ratio = np.log2(gamma_hat / gamma_true)
late_gene = np.flatnonzero(fate_genes)[np.argmax(t_on[fate_genes])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(t_on[fate_genes], ratio[fate_genes], s=12, color=CLASS_COLORS[0])
axes[0].axhline(0, color="k", ls="--", lw=1)
axes[0].set_xlabel("true switch-on time t_on (hidden)"); axes[0].set_ylabel("log2(γ′ / true γ/β)")
axes[0].set_title("fate genes: the later the switch, the steeper the fitted line")
g = late_gene
sc = axes[1].scatter(Ms[:, g], Mu[:, g], c=t_true, s=3, cmap="viridis")
smax = Ms[:, g].max()
axes[1].plot([0, smax], [0, gamma_true[g] * smax], "k--", lw=1.2, label="true γ/β")
axes[1].plot([0, smax], [0, gamma_hat[g] * smax], color=CLASS_COLORS[1], lw=1.5, label="fitted γ′")
axes[1].set_title(f"{var.index[g]}: switched on at t = {t_on[g]:.2f}"); axes[1].legend(fontsize=7, loc="upper left")
axes[1].set_xlabel("smoothed spliced"); axes[1].set_ylabel("smoothed unspliced"); plt.colorbar(sc, ax=axes[1], label="true time")
plt.suptitle("SIMULATED failure 1: genes far from steady state", y=1.02)
plt.tight_layout(); plt.show()
early_on, late_on = fate_genes & (t_on < 0.5), fate_genes & (t_on > 0.65)
print(f"🧪 Spearman(t_on, log2 ratio) among fate genes: {stats.spearmanr(t_on[fate_genes], ratio[fate_genes])[0]:.2f}")
print(f"median log2 ratio: genes switched on before t = 0.5: {np.median(ratio[early_on]):.2f}; "
      f"after t = 0.65: {np.median(ratio[late_on]):.2f}")
V_oracle = Mu - gamma_true * Ms                                   # same data, but the TRUE slope plugged in
sign_oracle = np.array([np.mean(np.sign(V_oracle[strong[:, g], g]) == np.sign(true_vel[strong[:, g], g]))
                        for g in np.flatnonzero(fate_genes)])
print(f"🧪 fate genes, sign correct where |v| is large: fitted γ′ {np.nanmedian(sign_ok[fate_genes]):.2f}, "
      f"true γ/β plugged in {np.median(sign_oracle):.2f}")

🔍 **What just happened?** Every fate gene's slope is overestimated, and the bias grows with the switch-on time
(Spearman 0.49; median log2 ratio 0.24 for genes switched on before $t = 0.5$, 0.62 after $t = 0.65$). The phase
portrait of the latest gene shows why: even the last cells lie well above the true line (dashed) — the gene is still
inducing when the experiment ends, so there are no cells at its upper steady state, and the fit through the "extreme"
cells is too steep. With degradation rates around 6 per unit time, a gene needs roughly half of the whole process to
settle, so in this simulation *no* fate gene fully reaches steady state. The last line isolates the cause: on the same
smoothed data, plugging in the true slope makes the sign right in essentially all cells where fate genes change
(median over fate genes 1.00), against 0.78 with the fitted slope. The bias of γ′ is the whole problem. scVelo's
**dynamical model** was designed for it: it fits the whole induction curve instead of assuming that the extremes are at
steady state (🚀 Bonus).

**Failure 2 — lineage-specific kinetics.** The model gives each gene *one* splicing rate and *one* degradation rate in
all cells. In real tissues a gene can be degraded faster in one lineage than in another (different RNA-binding proteins,
different cell sizes). Then there is no single steady-state line. We provoke this with `solve_kinetics`: one gene,
switched on at $t = 0.1$ in two lineages, with stable mRNA in lineage A ($\gamma = 4$) and unstable mRNA in lineage B
($\gamma = 16$), 500 cells per lineage sampled at random times and sequenced deeply (no smoothing needed).

In [ ]:
# ── 5.2 Provoking a failure: one gene, two lineages, two degradation rates ──
rng5 = np.random.default_rng(SEED + 5)
alpha_on = np.where(grid >= 0.1, 4.0, 0.0)                       # switched on at t = 0.1 in both lineages
kin = {"A": (*solve_kinetics(alpha_on, 12.0, 4.0, dt), 4.0),       # lineage A: stable mRNA, γ = 4
       "B": (*solve_kinetics(alpha_on, 12.0, 16.0, dt), 16.0)}     # lineage B: unstable mRNA, γ = 16
u_obs, s_obs, v_tru, lin, t_obs = [], [], [], [], []
for name, (u_k, s_k, gam) in kin.items():
    t_c = rng5.random(500)
    kk = np.minimum((t_c / dt).astype(int), len(grid) - 1)
    u_obs.append(rng5.poisson(200 * u_k[kk]) / 200); s_obs.append(rng5.poisson(200 * s_k[kk]) / 200)   # deep sequencing
    v_tru.append(12.0 * u_k[kk] - gam * s_k[kk]); lin += [name] * 500; t_obs.append(t_c)
u_obs, s_obs, v_tru, lin, t_obs = map(np.concatenate, (u_obs, s_obs, v_tru, [lin], t_obs))
g_fit, _ = fit_steady_state(u_obs[:, None], s_obs[:, None])
v_est = 12.0 * (u_obs - g_fit[0] * s_obs)                         # ×β so that it is comparable with ds/dt
print(f"fitted γ′ = {g_fit[0]:.3f}; true γ/β = {4 / 12:.3f} in lineage A and {16 / 12:.3f} in lineage B")
for name in "AB":
    late = (lin == name) & (t_obs > 0.8)
    moving = (lin == name) & (np.abs(v_tru) > 0.1 * np.abs(v_tru).max())
    print(f"lineage {name}: cells with t > 0.8 — mean true ds/dt {np.mean(v_tru[late]):+.3f}, mean estimated "
          f"{np.mean(v_est[late]):+.3f}; sign correct in cells that are really changing: "
          f"{np.mean(np.sign(v_est[moving]) == np.sign(v_tru[moving])):.0%}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for name, c in [("A", CLASS_COLORS[0]), ("B", CLASS_COLORS[4])]:
    m = lin == name
    ax.scatter(s_obs[m], u_obs[m], s=5, color=c, alpha=0.4, label=f"lineage {name} (γ = {kin[name][2]:.0f})")
    smax = s_obs[m].max()
    ax.plot([0, smax], [0, smax * kin[name][2] / 12], color=c, ls="--", lw=2, label=f"true steady-state line, {name}")
ax.plot([0, s_obs.max()], [0, g_fit[0] * s_obs.max()], color="k", lw=1.5, label="single fitted line γ′")
ax.set_xlabel("spliced"); ax.set_ylabel("unspliced"); ax.legend(fontsize=7)
ax.set_title("SIMULATED failure 2: two kinetic regimes, one fitted line"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The two lineages have different steady-state lines (dashed). The extreme-quantile fit picks
its "fully on" cells from lineage A, whose stable mRNA reaches the highest spliced levels, so the single fitted line
(γ′ = 0.38) lies close to lineage A's (0.33) — but slightly above it, because the last A cells are still inducing.
Both lineages now get wrong arrows. Lineage B's late cells are not changing at all (true ds/dt ≈ 0.00) but lie far
*above* the fitted line, so the model reports a large positive velocity (+2.9): it claims that lineage B is still
switching the gene on, forever. Lineage A's late cells are still slowly accumulating mRNA (true +0.25) but fall just
*below* the fitted line, so they get a negative velocity (−0.28), and the sign is wrong in 23 % of the A cells that
are really changing. Even with deep sequencing and no smoothing, a mis-specified model produces confident, wrong
arrows. Bergen *et al.* (2021) describe real examples of such multiple kinetic regimes, and of genes whose
transcription is boosted during differentiation, which can make velocity point **backwards** along a known trajectory.

**Other failure modes to remember** (none of which our simulation contains): the cell cycle and other repeated
processes break the assumption of one switch per gene; low unspliced counts make moments noisy, so velocity is
dominated by a few highly expressed genes; and the smoothing in 4.2 borrows information from neighbours, so arrows can
partly reflect the k-NN graph rather than the unspliced reads.

✏️ **Your turn.** Re-simulate the data of Section 1 with `depth=3` (very shallow sequencing) and re-run 4.2–4.6. How
much do the γ′ correlation and the forward fraction drop? Which genes stop being "velocity genes"?

---
## 6 · Optimal transport between time points

> 📍 **Where we are.** Velocity predicts the next few hours. A time course offers something else: real time. The cells
> of day $d+1$ are (mostly) the descendants of the cells of day $d$. If we assume that cells change as little as
> necessary between days, we can match the two populations — that is **optimal transport (OT)**, the idea of
> Waddington-OT (Schiebinger *et al.* 2019).

🧮 **Math refresher — optimal transport.** Picture day $d$ as piles of sand (one pile per cell, mass $a_i$) and day
$d+1$ as holes (mass $b_j$). Moving one unit of mass from $i$ to $j$ costs $C_{ij}$, here the squared distance between
the two cells in PC space.

- **Monge (1781)** asked for a *map* sending each pile to one hole. With cells this is too rigid: a progenitor may have
  descendants in both fates, and the two days need not have the same number of cells.
- **Kantorovich (1942)** allowed piles to be *split*: find a **coupling** matrix $\Pi$ with $\Pi_{ij} \ge 0$ the mass
  sent from $i$ to $j$,

$$\min_{\Pi \ge 0}\ \sum_{ij} C_{ij}\,\Pi_{ij} \quad\text{subject to}\quad \sum_j \Pi_{ij} = a_i,\quad \sum_i \Pi_{ij} = b_j .$$

*In words:* send all the mass out of every pile and fill every hole exactly, as cheaply as possible. This is a linear
programme; its solutions are sparse (most cells are matched to very few partners).

- **Entropic regularisation** (Cuturi 2013) adds a reward for spreading mass, with strength $\varepsilon$:

$$\min_{\Pi}\ \sum_{ij} C_{ij}\Pi_{ij} + \varepsilon\sum_{ij}\Pi_{ij}\,(\log\Pi_{ij} - 1) \quad\text{(same constraints)} .$$

*In words:* the second term is minus the entropy of the coupling, so small $\varepsilon$ gives nearly the sparse
Kantorovich solution and large $\varepsilon$ a blurry coupling in which every cell sends a little mass everywhere. For
cells this blur is a feature: we are uncertain about who descends from whom.

**Deriving Sinkhorn's algorithm.** Add Lagrange multipliers $f_i$ and $g_j$ for the two constraints and set the
derivative with respect to $\Pi_{ij}$ to zero:

$$C_{ij} + \varepsilon\log\Pi_{ij} - f_i - g_j = 0 \;\Rightarrow\; \Pi_{ij} = \underbrace{e^{f_i/\varepsilon}}_{x_i}\;\underbrace{e^{-C_{ij}/\varepsilon}}_{G_{ij}}\;\underbrace{e^{g_j/\varepsilon}}_{y_j} .$$

*In words:* the optimal coupling is a fixed matrix $G = e^{-C/\varepsilon}$ with its rows rescaled by numbers $x_i$ and
its columns by numbers $y_j$. The constraints then read $x_i\,(Gy)_i = a_i$ and $y_j\,(G^\top x)_j = b_j$, so we
alternate

$$x \leftarrow a \oslash (G\,y), \qquad y \leftarrow b \oslash (G^\top x)$$

*In words:* ($\oslash$ = element-wise division) rescale the rows so that every row sum is right, then the columns so
that every column sum is right, and repeat until both marginals are right. Each step fixes one constraint exactly; the
alternation converges (Sinkhorn's theorem). Two matrix–vector products per iteration — that is why entropic OT scales
to large data.

> 🔗 **Builds on** [Module 04](04_calculus_and_optimization.ipynb) (Lagrange multipliers for constrained optimisation)
> and [Module 13](13_mixture_models_em.ipynb): like EM, Sinkhorn alternates two simple updates, each of which solves
> half of the problem exactly.

In [ ]:
# ── 6.1 Sinkhorn from scratch, on a 1-D toy: one population splits into two ──
def sinkhorn(a, b, C, eps, n_iter=5000, tol=1e-9):
    """Entropic optimal transport by Sinkhorn scaling. Returns the coupling Π and the number of iterations used."""
    G = np.exp(-C / eps)
    x, y = np.ones(len(a)), np.ones(len(b))
    for it in range(1, n_iter + 1):
        x = a / (G @ y)
        y_new = b / (G.T @ x)
        if np.max(np.abs(y_new - y) / y_new) < tol:
            y = y_new
            break
        y = y_new
    return x[:, None] * G * y[None, :], it


rng6 = np.random.default_rng(SEED + 6)
src = np.sort(rng6.normal(0, 0.6, 40))                                   # "day d": one population
dst = np.sort(np.r_[rng6.normal(-1.5, 0.3, 40), rng6.normal(1.5, 0.3, 40)])   # "day d+1": two branches
C_toy = (src[:, None] - dst[None, :]) ** 2
C_toy /= np.median(C_toy)
a_toy, b_toy = np.full(40, 1 / 40), np.full(80, 1 / 80)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, eps in zip(axes, [0.005, 0.05, 0.5]):
    Pi, it = sinkhorn(a_toy, b_toy, C_toy, eps)
    rows_p = Pi / Pi.sum(axis=1, keepdims=True)
    partners = np.exp(-(rows_p * np.log(rows_p + 1e-300)).sum(axis=1)).mean()   # effective number of partners
    ax.imshow(Pi, aspect="auto", cmap="magma_r", extent=[dst[0], dst[-1], src[-1], src[0]])
    ax.set_title(f"ε = {eps}: {it} iterations, ≈ {partners:.0f} partners per cell")
    ax.set_xlabel("day d+1 position"); ax.set_ylabel("day d position")
    print(f"ε = {eps}: max marginal error {max(np.abs(Pi.sum(1) - a_toy).max(), np.abs(Pi.sum(0) - b_toy).max()):.1e}")
plt.suptitle("Toy couplings (SIMULATED 1-D cells): small ε = sharp matching, large ε = blur", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** One population on day $d$ is matched to a population that has split into two branches on
day $d+1$. With small ε the coupling is nearly a sharp, monotone matching: the left half of the source goes to the left
branch and the right half to the right branch, each cell to about a dozen of the 80 targets. As ε grows the coupling
blurs; at ε = 0.5 each source cell spreads its mass over about 57 targets, including both branches. Both marginals
are satisfied to the tolerance in every case; smaller ε needs many more iterations. Notice the *assumption* hidden in OT: cells on the left go left because that is
cheaper. If the real choice of branch were unrelated to position, OT would still produce a confident-looking matching —
keep that in mind for Section 6.5.

### Coupling the sampling days

Now the real thing: day 0 → 1, 1 → 2, 2 → 3, in the 10-dimensional PC space, each cell with equal mass, costs divided
by their median so that one ε fits all pairs of days.

In [ ]:
# ── 6.2 Couplings between consecutive days in PC space ──
def day_couplings(Zc, day, eps=0.05):
    """Sinkhorn couplings between consecutive sampling days; returns index lists and row-normalised couplings."""
    idx = [np.flatnonzero(day == d) for d in range(day.max() + 1)]
    trans = []
    for d in range(len(idx) - 1):
        A, B = Zc[idx[d]], Zc[idx[d + 1]]
        C = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)
        Pi, it = sinkhorn(np.full(len(A), 1 / len(A)), np.full(len(B), 1 / len(B)), C / np.median(C), eps)
        trans.append(Pi / Pi.sum(axis=1, keepdims=True))     # row i: where the mass of cell i goes
    return idx, trans


idx_day, trans = day_couplings(Z10, day)
print("cells per day:", [len(i) for i in idx_day])
# 🧪 sanity checks against the hidden truth
for d in range(3):
    src_i, dst_i = idx_day[d], idx_day[d + 1]
    fwd = (trans[d] @ t_true[dst_i]) - t_true[src_i]          # expected time gained by each cell's descendants
    committed = state[src_i] != "progenitor"
    line = f"🧪 day {d} → {d + 1}: descendants later in true time for {np.mean(fwd > 0):.0%} of cells"
    if committed.sum() >= 20:
        own = (trans[d][committed] * (fate[src_i][committed][:, None] == fate[dst_i][None, :])).sum(axis=1)
        line += f"; {committed.sum()} committed cells send {own.mean():.1%} of their mass to their own fate"
    print(line)

🔍 **What just happened?** Each coupling is a matrix of about 500 × 500 entries, computed in well under a second. The
🧪 checks look sensible. For every cell, the expected true time of its inferred descendants is later than its own.
Cells committed on day 2 send 99.5 % of their mass to cells of the same fate on day 3. Cells committed on day 1 send
only 75 %: many of them branched moments ago and still look like progenitors, so OT cannot yet tell which arm they
are on — and splits its bet.

### Ancestors, descendants and fate probabilities

The row-normalised couplings are transition matrices from one day to the next. Multiplying them **chains** the days:
$T_{0\to3} = T_{0\to1}T_{1\to2}T_{2\to3}$ gives, for every day-0 cell, the distribution of its descendants on day 3.
On the last day the two fates are clearly separated (we label them with k-means, $k = 2$, on day 3 alone — no hidden
truth used), so a cell's **fate probability** is the share of its descendants that land in fate A:

$$p_A(i) = \big(T_{d\to d+1}\cdots T_{2\to3}\,\mathbf 1_A\big)_i .$$

*In words:* follow the mass of cell $i$ forward day by day; $p_A(i)$ is how much of it ends up among the fate-A cells
of day 3. Run backwards (with the columns), the same couplings give **ancestors**: which earlier cells sent mass to a
chosen group.

In [ ]:
# ── 6.3 Fate probabilities by chaining couplings, and descendant maps ──
def ot_fate_probabilities(Zc, day, eps=0.05):
    """Fate-A probability of every cell by chaining Sinkhorn couplings to the last day (labelled by 2-means)."""
    idx, tr = day_couplings(Zc, day, eps)
    last = idx[-1]
    lab = KMeans(2, n_init=10, random_state=SEED).fit_predict(Zc[last])
    p = np.zeros(len(Zc))
    p[last] = (lab == 0).astype(float)
    vec = p[last]
    for d in range(len(idx) - 2, -1, -1):
        vec = tr[d] @ vec
        p[idx[d]] = vec
    return p, last, lab


p_ot, last, lab_last = ot_fate_probabilities(Z10, day)
if np.mean((lab_last == 0) == (fate[last] == "A")) < 0.5:          # name the k-means clusters after the truth
    p_ot, lab_last = 1 - p_ot, 1 - lab_last
print(f"🧪 day-3 k-means labels agree with the hidden fate for {np.mean((lab_last == 0) == (fate[last] == 'A')):.1%} of cells")
print(f"progenitors: mean p_A = {p_ot[state == 'progenitor'].mean():.3f} (sd {p_ot[state == 'progenitor'].std():.3f}); "
      f"share of fate-A cells on day 3: {np.mean(lab_last == 0):.3f}")

In [ ]:
# ── 6.4 Descendants of single cells ──
def descendants(cell, d0):
    """Descendant mass of one cell of day d0 on every later day, relative to a uniform spread (1 = average)."""
    out = np.full(len(Z10), np.nan)
    mass = trans[d0][np.flatnonzero(idx_day[d0] == cell)[0]]
    for d in range(d0 + 1, 4):
        out[idx_day[d]] = mass * len(idx_day[d])
        if d < 3:
            mass = mass @ trans[d]
    return out


picks_ot = {"day-0 progenitor": idx_day[0][np.argmin(np.abs(t_true[idx_day[0]] - 0.15))],
            "day-1 cell committed to A": idx_day[1][np.argmin(np.abs(t_true[idx_day[1]] - 0.5)
                                                              + 9 * (state[idx_day[1]] != "fate A"))]}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
plot_embedding(E2, p_ot, ax=axes[0], s=3, cmap="coolwarm_r", title="OT fate probability p_A (blue = fate A)")
for ax, (name, cell) in zip(axes[1:], picks_ot.items()):
    dsc = descendants(cell, day[cell])
    ax.scatter(*E2.T, s=2, color="0.85")
    later = np.isfinite(dsc)
    sc = ax.scatter(*E2[later].T, c=np.minimum(dsc[later], 3), s=3, cmap="viridis", vmin=0, vmax=3)
    ax.scatter(*E2[cell], s=150, marker="*", color="k")
    ax.set_title(f"descendants of a {name} ★"); ax.set_xticks([]); ax.set_yticks([])
    print(f"{name} ★: {np.nansum(dsc[last][lab_last == 0]) / len(last):.2f} of its day-3 mass in fate A "
          f"(hidden fate: {fate[cell]})")
plt.colorbar(sc, ax=axes[2], label="descendant mass (1 = uniform)")
plt.suptitle("SIMULATED: optimal-transport fate mapping (grey = earlier days)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Once cells have branched, their fate probabilities are close to 1 (blue, the fate-A arm) or 0
(red, the fate-B arm).
All progenitors — on day 0 and the uncommitted ones on day 1 — sit just below 0.5: OT says "either fate", with odds set
by the sizes of the two day-3 populations (242 A vs 258 B cells). The descendant maps make it concrete: the day-0
progenitor's descendants are spread almost uniformly over both arms (0.49 of its day-3 mass in A), whereas the
descendants of a day-1 cell already committed to A stay on the A arm (1.00). With equal masses, the *ancestors* of a
group (computed with the transposed couplings) are proportional to the fate probabilities towards it, so the left map
is also an ancestor map of fate A. Is the progenitors' 0.5 a failure of the method? The 🧪 check decides.

In [ ]:
# ── 6.5 🧪 Can the fate of a progenitor be predicted? AUROC against the hidden fate, without and with priming ──
def fate_auroc_table(p, day, state, fate):
    """AUROC of the fate-A probability for the true fate, per group of cells."""
    groups = {"day-0 progenitors": (day == 0) & (state == "progenitor"), "day-1 progenitors": (day == 1) & (state == "progenitor"),
              "day-1 committed": (day == 1) & (state != "progenitor"), "day-2 cells": day == 2}
    return {name: roc_auc_score(fate[m] == "A", p[m]) for name, m in groups.items()}


results = {"priming = 0 (our data)": fate_auroc_table(p_ot, day, state, fate)}
for priming in [0.005, 0.02, 0.1, 0.6]:
    sim = simulate_differentiation(np.random.default_rng(SEED + 1), n_cells=2000, n_genes=200, n_days=4, priming=priming)
    o = sim["obs"]
    Zs = pca(log_normalize(sim["spliced"]), n_components=10)[0]
    p_s, last_s, lab_s = ot_fate_probabilities(Zs, o.day.to_numpy())
    if np.mean((lab_s == 0) == (o.fate.to_numpy(str)[last_s] == "A")) < 0.5:
        p_s = 1 - p_s
    results[f"priming = {priming}"] = fate_auroc_table(p_s, o.day.to_numpy(), o.state.to_numpy(str), o.fate.to_numpy(str))
    if priming == 0.6:
        p_primed, t_primed, f_primed = p_s, o.time.to_numpy(), o.fate.to_numpy(str)
print("🧪 AUROC of the OT fate probability for the true fate (0.5 = chance, 1 = perfect):")
print(pd.DataFrame(results).T.round(3).to_string())

In [ ]:
# ── 6.6 Fate probabilities without and with priming ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, p, tt, ff, title in [(axes[0], p_ot, t_true, fate, "priming = 0 (our data)"),
                             (axes[1], p_primed, t_primed, f_primed, "priming = 0.6")]:
    for f, c in [("A", CLASS_COLORS[0]), ("B", CLASS_COLORS[4])]:
        m = (ff == f) & (tt < 0.99)
        ax.scatter(tt[m], p[m], s=3, alpha=0.5, color=c, label=f"true fate {f}")
    ax.axvline(0.35, color="k", ls="--", lw=1); ax.text(0.36, 0.5, "branch point", fontsize=7, rotation=90, va="center")
    ax.set_xlabel("true latent time (hidden)"); ax.set_title(title)
axes[0].set_ylabel("OT fate probability p_A"); axes[0].legend(markerscale=4, fontsize=7)
plt.suptitle("SIMULATED: before the branch, fate is predictable only if it is written in the RNA", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** This is the most important result of the notebook. In our data (no priming), the fate
probabilities of progenitors are useless: AUROC 0.49 on day 0 and 0.53 on day 1, i.e. chance — while committed cells
are classified almost perfectly (0.96 on day 1, 1.0 on day 2). That is not a weakness of OT. In this simulation the two
lineages are **identical** before the branch point: the information that decides a progenitor's fate is simply not in
its RNA, and no method — OT, velocity or a deep network — can recover it. With **priming**, fate genes are weakly
transcribed in the "right" lineage before the branch (from $t = 0.1$ on, at a fraction `priming` of their full rate).
Then the progenitors' transcriptomes differ slightly, and the same OT pipeline detects it: even with priming 0.005 —
half a per cent of the full transcription rate — the day-1 progenitors are well predicted (AUROC 0.97), and from 0.02
on almost perfectly. Day-0 progenitors stay harder (AUROC 0.57–0.76), because many of them were sampled before priming
starts ($t < 0.1$). The scatter plots show it directly: without priming, $p_A$ is flat at 0.48 for every cell before
the branch point and splits only afterwards; with priming it splits at $t \approx 0.12$. Such a weak signal is visible
here only because simulated fate genes are otherwise *exactly* silent; in real cells, background transcription and
measurement noise would hide much of it.

**The lesson.** "Fate is unpredictable from the transcriptome at this stage" and "the method failed" look the same in a
real data set. Only lineage tracing (Weinreb *et al.* 2020) can tell them apart, and it suggested that real progenitors
sit somewhere in between: their fate is partly, not fully, readable from their RNA.

⚠️ **OT assumptions.** We gave every cell the same mass, so OT assumed no cell divides or dies between days. If fate-A
cells proliferate faster, balanced OT must send extra progenitor mass to A to fill the larger A population, biasing the
fate probabilities. Waddington-OT therefore uses **unbalanced** OT with estimated growth rates (🚀 Bonus).

✏️ **Your turn.** Recompute `p_ot` with `eps=0.005` and with `eps=0.5` (argument of `ot_fate_probabilities`). Do the
committed cells' probabilities stay near 0 and 1? Does any value of ε make the unprimed progenitors predictable?

### 📄 Research connection — Optimal transport and fate mapping

- **What the field does.** Schiebinger *et al.* (2019) introduced **Waddington-OT**: densely sampled time courses of
    reprogramming fibroblasts, unbalanced entropic OT between consecutive time points with growth rates estimated from
    proliferation and apoptosis genes, and ancestor/descendant analyses that identified regulators of the successful
    path. Entropic OT itself rests on Cuturi (2013), who showed that Sinkhorn scaling makes OT fast enough for machine
    learning; Peyré & Cuturi (2019) is the standard reference. Weinreb *et al.* (2020) combined heritable barcodes with
    scRNA-seq in haematopoiesis, providing ground truth against which fate-prediction methods could be tested.
- **What this notebook builds.** Sinkhorn from scratch, couplings between all consecutive days, chained fate probabilities and descendant
    maps, graded against the hidden fate of every cell — including the negative result that unprimed progenitors are
    unpredictable and the positive result that priming makes them predictable.
- **Where the frontier is.** OT is now a general toolkit for single-cell data: moscot applies it to time courses, spatial alignment
    of tissue sections and mapping cells onto space; neural OT and flow-matching models
    ([Module 27](27_diffusion_flow_matching.ipynb)) learn continuous dynamics instead of discrete couplings; and OT
    predicts responses to perturbations by matching treated and control cells. Open questions: how to choose costs
    and ε, how to model growth, and how to quantify the uncertainty of an inferred ancestry.
- **Read next.** Schiebinger *et al.* 2019 (*Cell*), *Optimal-transport analysis of single-cell gene expression identifies developmental trajectories in reprogramming*; Weinreb *et al.* 2020 (*Science*), *Lineage tracing on transcriptional landscapes links state to fate during differentiation*.

---
## 7 · Markov-chain fate mapping (the CellRank idea)

> 📍 **Where we are.** OT needs several time points. With a single snapshot we can still ask where cells are headed, by
> letting the velocity-directed Markov chain of Section 4 run until it reaches a terminal state. This is the idea of
> **CellRank** (Lange *et al.* 2022) and, with pseudotime instead of velocity, of Palantir (Setty *et al.* 2019).

🧮 **Math refresher — absorbing Markov chains.** Make the terminal cells **absorbing**: once the walker arrives, it
stays. Order the cells as transient ($Q$ block) and absorbing (identity block):

$$T = \begin{pmatrix} Q & R \\ 0 & I \end{pmatrix}, \qquad B = (I - Q)^{-1} R .$$

*In words:* $Q$ holds the transition probabilities between non-terminal cells and $R$ those from non-terminal cells
into the terminal ones. $B_{ia}$ is the probability that a walker started at cell $i$ is eventually absorbed in terminal
state $a$. Why? By **first-step analysis**, the walker either jumps straight into $a$ (probability $R_{ia}$) or to a
transient cell $j$ and continues from there: $B = R + QB$, whose solution is $B = (I-Q)^{-1}R$. With several absorbing
cells per fate, we add up the columns of each fate.

> 🔗 **Builds on** [Module 19](19_hmm_inference.ipynb): the forward algorithm sums over paths step by step;
> $(I-Q)^{-1} = I + Q + Q^2 + \dots$ sums over paths of *every* length at once.

We combine the velocity transitions $\pi$ with a small amount of undirected k-NN diffusion (weights 0.8 and 0.2), which
keeps the chain connected where velocity is noisy. Terminal states: on day 3, the 30 cells of each k-means cluster that
lie farthest from the day-0 cells.

In [ ]:
# ── 7.1 Absorption probabilities of a velocity-directed Markov chain ──
def absorption_fate(P_vel, nbr, term_A, term_B, w_vel=0.8):
    """Probability of absorption in the terminal cells of fate A for a chain mixing velocity transitions (weight w_vel)
    with undirected k-NN diffusion. Solves (I − Q) B = R with a dense linear solve."""
    n, k = P_vel.shape[0], nbr.shape[1]
    rows = np.repeat(np.arange(n), k)
    T_vel = sparse.csr_matrix((P_vel.ravel(), (rows, nbr.ravel())), shape=(n, n))
    A = sparse.csr_matrix((np.ones(n * k), (rows, nbr.ravel())), shape=(n, n))
    A = A.maximum(A.T)
    T = (w_vel * T_vel + (1 - w_vel) * sparse.diags(1 / np.asarray(A.sum(1)).ravel()) @ A).toarray()
    absorbing = np.r_[term_A, term_B]
    transient = np.setdiff1d(np.arange(n), absorbing)
    Q, R = T[np.ix_(transient, transient)], T[np.ix_(transient, absorbing)]
    R2 = np.c_[R[:, :len(term_A)].sum(1), R[:, len(term_A):].sum(1)]    # into fate A, into fate B
    Bm = np.linalg.solve(np.eye(len(transient)) - Q, R2)
    p = np.zeros(n); p[transient] = Bm[:, 0]; p[term_A] = 1.0
    return p, Bm.sum(axis=1)


dist0 = np.linalg.norm(Z10 - Z10[day == 0].mean(axis=0), axis=1)
term = {c: last[lab_last == c][np.argsort(dist0[last[lab_last == c]])[-30:]] for c in (0, 1)}   # 0 = fate A
t0 = time.time()
p_vel, total = absorption_fate(P_est, nbr, term[0], term[1], w_vel=0.8)
p_diff, _ = absorption_fate(P_est, nbr, term[0], term[1], w_vel=0.0)   # undirected diffusion only
print(f"absorption probabilities in {time.time() - t0:.1f} s; every walker is absorbed: min total = {total.min():.4f}")
cmp = {"velocity chain (0.8/0.2)": fate_auroc_table(p_vel, day, state, fate),
       "undirected diffusion only": fate_auroc_table(p_diff, day, state, fate),
       "optimal transport (6.3)": fate_auroc_table(p_ot, day, state, fate)}
print("🧪 AUROC for the true fate:")
print(pd.DataFrame(cmp).T.round(3).to_string())
prog = state == "progenitor"
print(f"progenitors: velocity chain p_A = {p_vel[prog].mean():.3f} (sd {p_vel[prog].std():.3f}), "
      f"diffusion {p_diff[prog].mean():.3f} (sd {p_diff[prog].std():.3f}), OT {p_ot[prog].mean():.3f} (sd {p_ot[prog].std():.3f})")
comm = (state != "progenitor") & (day < 3)
print("🧪 mean probability given to the true fate, committed cells of days 1–2:",
      {k: round(float(np.mean(np.where(fate[comm] == "A", p[comm], 1 - p[comm]))), 3)
       for k, p in [("velocity", p_vel), ("diffusion", p_diff), ("OT", p_ot)]})

In [ ]:
# ── 7.2 Fate probabilities from the Markov chain vs optimal transport ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
plot_embedding(E2, p_vel, ax=axes[0], s=3, cmap="coolwarm_r", title="velocity chain: P(absorbed in A)")
plot_embedding(E2, p_diff, ax=axes[1], s=3, cmap="coolwarm_r", title="undirected diffusion: P(absorbed in A)")
for ax in axes[:2]:
    ax.scatter(*E2[term[0]].T, s=12, marker="x", color="k"); ax.scatter(*E2[term[1]].T, s=12, marker="x", color="k")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
for s_, c in [("progenitor", "0.5"), ("fate A", CLASS_COLORS[0]), ("fate B", CLASS_COLORS[4])]:
    m = (state == s_) & (day < 3)
    axes[2].scatter(p_ot[m], p_vel[m], s=4, color=c, alpha=0.6, label=s_)
axes[2].set_xlabel("optimal transport p_A"); axes[2].set_ylabel("velocity chain p_A"); axes[2].legend(markerscale=3, fontsize=7)
axes[2].set_title("two methods, same ranking (days 0–2)")
plt.suptitle("SIMULATED: fate probabilities as absorption probabilities (× = terminal cells)", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The absorption probabilities agree with OT on what matters, although they use different
information (one snapshot plus velocity, versus four time points and no velocity): committed cells get probabilities
near 0 or 1, and every progenitor gets the *same* value, so the AUROC for progenitors is again ≈ 0.5 — the same honest
"unknown". The *level* differs: the chain gives progenitors $p_A \approx 0.69$, OT about 0.48, while the true split is
50:50. Absorption probabilities depend on the geometry of the graph near the branch point and on which cells we declared
terminal, so they rank cells well without being calibrated probabilities (AUROC measures ranking only;
[Module 42](42_trustworthy_ml.ipynb) treats calibration). The comparison with undirected diffusion shows what the
*direction* adds: without velocity the walker wanders back and forth across the branch point, so even committed cells
receive diluted probabilities (mean probability for their true fate 0.70, against 0.93 with velocity and 0.91 with OT),
although their ranking stays good. This is why CellRank builds its chain from velocity, or from pseudotime with a
penalty for going backwards (Palantir's idea).

✏️ **Your turn.** Try `w_vel=0.5` and `w_vel=1.0` in `absorption_fate`. With `w_vel=1.0` the chain follows velocity
only; does every walker still get absorbed (look at `total.min()`)? What happens to cells whose velocity points away
from both terminal states?

---
## 8 · 🧭 From toy to real data

> 📍 **Where we are.** Every method above ran on 2,000 simulated cells whose genes obey the velocity equations exactly.
> Here is what changes with real data — no downloads, a map for your first real analysis.

**Scale and preprocessing.** Real experiments have tens of thousands of genes and from thousands to millions of cells.
Spliced/unspliced matrices are produced from the raw reads with [velocyto](http://velocyto.org/), STARsolo or
kallisto | bustools; the choice of counting tool and of which reads are "ambiguous" changes the unspliced counts
noticeably. Time courses need careful experimental design: enough cells per time point, time points close enough
for the populations to overlap, and every time point processed in a way that avoids confounding batch with time
([Module 31](31_batch_integration.ipynb)).

**Public data to practise on.** The [scVelo documentation](https://scvelo.readthedocs.io/) and
[CellRank documentation](https://cellrank.readthedocs.io/) ship tutorial data sets (pancreatic endocrine development,
dentate gyrus neurogenesis); Schiebinger *et al.*'s reprogramming time course accompanies
[Waddington-OT](https://broadinstitute.github.io/wot/); Weinreb *et al.*'s lineage-traced haematopoiesis data are a
rare case where true clonal fates are known; [CELLxGENE](https://cellxgene.cziscience.com/) hosts many developmental
atlases.

**Production tools.** scVelo (steady-state, stochastic and dynamical velocity models), CellRank (Markov chains from
velocity, pseudotime, real time or OT; terminal-state detection; fate probabilities and driver genes),
[moscot](https://moscot.readthedocs.io/) and Waddington-OT (optimal transport for time courses), Scanpy's `diffmap`,
`dpt` and `paga`, [Monocle 3](https://cole-trapnell-lab.github.io/monocle3/), Slingshot and Palantir.

**What goes wrong first.**

1. **Velocity that contradicts known biology.** Arrows pointing backwards along a well-known trajectory usually mean
   that some genes violate the model (Section 5); inspect the phase portraits of the top velocity genes, try the
   dynamical model, and never trust arrows you cannot explain gene by gene.
2. **Spurious trajectories.** Pseudotime and MSTs always produce an ordering and a tree, even across unrelated cell
   types or between batches. Check connectivity (PAGA), use known markers and time points, and treat a trajectory as a
   hypothesis.
3. **Growth, death and sampling.** OT assumes that the cell populations of consecutive time points are comparable
   samples. Proliferation differences, cell death, dissociation biases and different numbers of cells per time point
   all distort the couplings unless they are modelled (unbalanced OT, growth rates).

**What simulation cannot teach.** Real data have no `obs.time` or `obs.fate` column. The closest substitutes for our 🧪
checks are lineage tracing (heritable barcodes: Weinreb *et al.* 2020), metabolic labelling of newly transcribed RNA
(which measures RNA age directly), real time points, and perturbations that block one branch.

## 📌 Key takeaways

- **Single-cell data are snapshots**: order, direction and fate must be inferred. Spliced and unspliced reads from an
  ordinary experiment add a short-term arrow; sampling days add real time; lineage barcodes add true ancestry.
- **Pseudotime** is a distance along the data manifold from a root: a random walk on the k-NN graph (a Markov chain),
  its slowly decaying eigenvectors (diffusion components) and the diffusion pseudotime ordered our cells with ρ ≈ 0.98.
  It needs a root from outside, and a tree over clusters always exists — evidence for a lineage needs connectivity.
- **The kinetic model** $du/dt = \alpha - \beta u$, $ds/dt = \beta u - \gamma s$ puts steady-state cells on the line
  $u = (\gamma/\beta)s$; cells above it are inducing, cells below it repressing. Steady-state velocity $u - \gamma' s$
  correlated well with the true velocity, and its projection pointed forward in time for 86 % of cells.
- **Velocity fails when its assumptions fail**: genes still inducing at the end bias γ′; lineage-specific kinetics
  produce confident wrong arrows; real data add bursts, boosts, cycles and sparse unspliced counts.
- **Entropic optimal transport** couples consecutive time points; Sinkhorn alternately rescales the rows and columns of
  $e^{-C/\varepsilon}$; chaining couplings gives ancestors, descendants and fate probabilities.
- **Fate can be invisible.** Without priming, progenitor fate was unpredictable (AUROC ≈ 0.5) by OT and by Markov
  chains alike, because the information was not in the RNA; with priming the same pipeline found it. A null result can
  be biology, not method failure.
- **Absorption probabilities** of a velocity-directed Markov chain, $B = (I-Q)^{-1}R$, give fate probabilities from a
  single snapshot; direction sharpens them compared with undirected diffusion.

---
## ✅ Checkpoint

1. Show that at steady state $u = (\gamma/\beta)\,s$. What does it mean if a cell lies far above this line?
   <details><summary>Answer</summary>Setting $du/dt = 0$ gives $u^\ast = \alpha/\beta$, and $ds/dt = 0$ gives $s^\ast = \beta u^\ast/\gamma = \alpha/\gamma$; dividing, $u^\ast/s^\ast = \gamma/\beta$, independent of $\alpha$. A cell far above the line has more unspliced RNA than steady state predicts: the gene was recently switched on (or up), and $ds/dt = \beta(u - \tfrac{\gamma}{\beta}s) > 0$.</details>
2. Why does pseudotime need a root, and name two ways to choose one.
   <details><summary>Answer</summary>Diffusion distances are symmetric, so the graph cannot distinguish the start from the end of a process. The root must come from outside: the earliest sampling day (as here), known stem/progenitor markers, or RNA velocity (cells from which arrows flow away; CellRank and scVelo can estimate initial states).</details>
3. The transition matrix $T = D^{-1}K$ always has eigenvalue 1. Why, and what is the corresponding eigenvector?
   <details><summary>Answer</summary>Each row of $T$ sums to 1, so $T\mathbf 1 = \mathbf 1$: the constant vector is an eigenvector with eigenvalue 1. It carries no information about the shape of the data, which is why diffusion components start at $\psi_1$.</details>
4. In Sinkhorn's algorithm, what happens to the coupling as $\varepsilon \to 0$ and as $\varepsilon \to \infty$?
   <details><summary>Answer</summary>As $\varepsilon \to 0$ the coupling approaches the (sparse) unregularised Kantorovich solution, but $e^{-C/\varepsilon}$ underflows and convergence becomes slow. As $\varepsilon \to \infty$ the entropy term dominates and the coupling tends to the independent coupling $a b^\top$: every source sends mass everywhere in proportion to the targets.</details>
5. Our OT fate probabilities for unprimed progenitors had AUROC ≈ 0.5. Give two different explanations a researcher would have to distinguish on real data, and how.
   <details><summary>Answer</summary>(1) The fate is not determined (or not yet encoded) in the measured transcriptome — biology; (2) the method or its assumptions failed (wrong cost, ε, unmodelled growth, too few time points). Lineage tracing (clonal barcodes) gives true fates to test against; in our simulation the truth showed it was (1), since primed data were predicted well by the same pipeline.</details>
6. Why is the steady-state estimate of γ′ too large for a gene switched on late in the experiment?
   <details><summary>Answer</summary>The fit assumes the cells with the highest u and s sit at the upper steady state. If the gene is still inducing at the end, those cells lie above the steady-state line (excess unspliced RNA), so the least-squares slope through them is too steep; velocities $u - \gamma' s$ are then too small and some have the wrong sign.</details>
7. Compute the absorption probabilities by hand: transient cells 1 and 2, terminal states A and B. From 1: to 2 with 0.5, to A with 0.5. From 2: to 1 with 0.5, to B with 0.5.
   <details><summary>Answer</summary>First-step analysis: $b_1 = 0.5 + 0.5\,b_2$ and $b_2 = 0.5\,b_1$ (probabilities of reaching A). Substituting, $b_1 = 0.5 + 0.25\,b_1$, so $b_1 = 2/3$ and $b_2 = 1/3$. Matrix form: $Q = \begin{pmatrix}0 & 0.5\\ 0.5 & 0\end{pmatrix}$, $R_A = (0.5, 0)^\top$, $(I-Q)^{-1}R_A = (2/3, 1/3)^\top$.</details>
8. Reading code: in `velocity_transitions`, what does `D = Ms[nbr] - Ms[:, None, :]` contain, and why do we compare it with the velocity through a cosine rather than a dot product?
   <details><summary>Answer</summary>D[i, j] is the displacement in (smoothed spliced) expression space from cell i to its j-th neighbour. The cosine measures only the *direction* agreement between that displacement and the cell's velocity; a dot product would favour distant neighbours and cells with large velocity vectors, mixing speed and distance into the probabilities.</details>

## 🧠 Questions & discussion

1. Our velocity worked in 86 % of cells because the simulator obeys the model. List three properties of real transcription that the simulator lacks, and predict how each would affect the arrows.
2. Pseudotime orders cells, but is pseudotime *time*? Give an example in which two cells with the same pseudotime are at very different real times (hint: the speed of differentiation can differ between lineages).
3. Weinreb *et al.* (2020) found that fate was only partly predictable from the transcriptome. What kinds of information could decide fate without appearing in mRNA counts?
4. OT assumes that cells change as little as possible between time points. When is this biologically wrong (think of fast responses to a stimulus, or cells migrating into the tissue between samples)?
5. Trajectory figures with confident arrows appear in many papers and grant proposals. What evidence would you require before accepting a claim such as 'cluster 5 gives rise to cluster 9' — and what is lost if such claims are made too easily, for example in studies of disease progression in patients?

## 🚀 Bonus — going further

- **Dynamical model on one gene.** For a fate gene, fit α, γ (with β = 1 after rescaling) and a switch time by alternating between assigning each cell a latent time on the model curve (E-step-like) and refitting the rates by least squares (M-step-like) — the idea of scVelo's dynamical model (Bergen *et al.* 2020). Does it remove the γ′ bias of Section 5?
- **Unbalanced OT with growth.** Give fate-A cells 1.5× more mass on later days (as if they proliferated faster) and recompute the fate probabilities with balanced OT. Then implement unbalanced Sinkhorn (replace the marginal constraints by KL penalties: the scaling updates get an exponent λ/(λ+ε)) and compare.
- **Palantir-style entropy.** Compute the entropy $-p_A\log p_A - (1-p_A)\log(1-p_A)$ of the fate probabilities along pseudotime: where is the 'decision' made, and how does that change with priming?
- **Velocity pseudotime.** Find the stationary distribution of the *reverse* velocity chain to locate initial states without using the sampling days, and compare it with the root chosen in Section 2.
- **Try the real thing.** Install scVelo and CellRank, run their pancreas tutorial, and compare the phase portraits of the top velocity genes with those of Section 4.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 32 — Cell dynamics: pseudotime, RNA velocity and optimal transport       Date:
Biological question: in what order, in which direction and towards which fate do progenitors differentiate?
Data (SIMULATED): simulate_differentiation (2,000 cells × 200 genes, 4 days; truth: latent time, fate, α, β, γ,
                  true velocity); priming = 0 and 0.005–0.6 for the fate-predictability test
Method: diffusion map + DPT; k-means + MST / PAGA-like tree; steady-state RNA velocity + velocity graph;
        Sinkhorn OT between days; absorbing Markov chain (CellRank idea)
Key equation: du/dt = α − βu, ds/dt = βu − γs;  Π = diag(x) e^{−C/ε} diag(y);  B = (I − Q)^{-1} R
Key parameters: k of the k-NN graph; root cell; quantile for γ′; κ of the velocity graph; ε; terminal states
Ground-truth check result: DPT ρ = …; γ′ ρ = …; forward fraction …; AUROC progenitors (priming 0 / 0.6) …
What changed when I modified …: (e.g. root, κ, ε, depth, w_vel)
One failure mode: (e.g. late-switching genes bias γ′; lineage-specific kinetics; unpredictable fate)
Connection to research (paper/tool): DPT/PAGA (Scanpy), scVelo, CellRank, Waddington-OT/moscot, Weinreb 2020
```

## 📚 References

- Trapnell C. *et al.* (2014) The dynamics and regulators of cell fate decisions are revealed by pseudotemporal ordering of single cells. *Nature Biotechnology* 32:381–386.
- Coifman R. R., Lafon S. (2006) Diffusion maps. *Applied and Computational Harmonic Analysis* 21:5–30.
- Haghverdi L., Büttner M., Wolf F. A., Buettner F., Theis F. J. (2016) Diffusion pseudotime robustly reconstructs lineage branching. *Nature Methods* 13:845–848.
- Wolf F. A. *et al.* (2019) PAGA: graph abstraction reconciles clustering with trajectory inference through a topology preserving map of single cells. *Genome Biology* 20:59.
- Saelens W., Cannoodt R., Todorov H., Saeys Y. (2019) A comparison of single-cell trajectory inference methods. *Nature Biotechnology* 37:547–554.
- La Manno G. *et al.* (2018) RNA velocity of single cells. *Nature* 560:494–498.
- Bergen V., Lange M., Peidli S., Wolf F. A., Theis F. J. (2020) Generalizing RNA velocity to transient cell states through dynamical modeling. *Nature Biotechnology* 38:1408–1414.
- Bergen V., Soldatov R. A., Kharchenko P. V., Theis F. J. (2021) RNA velocity — current challenges and future perspectives. *Molecular Systems Biology* 17:e10282.
- Gorin G., Fang M., Chari T., Pachter L. (2022) RNA velocity unraveled. *PLoS Computational Biology* 18:e1010492.
- Cuturi M. (2013) Sinkhorn distances: lightspeed computation of optimal transport. *Advances in Neural Information Processing Systems (NeurIPS)* 26.
- Peyré G., Cuturi M. (2019) Computational optimal transport. *Foundations and Trends in Machine Learning* 11:355–607.
- Schiebinger G. *et al.* (2019) Optimal-transport analysis of single-cell gene expression identifies developmental trajectories in reprogramming. *Cell* 176:928–943.
- Lange M. *et al.* (2022) CellRank for directed single-cell fate mapping. *Nature Methods* 19:159–170.
- Weinreb C., Rodriguez-Fraticelli A., Camargo F. D., Klein A. M. (2020) Lineage tracing on transcriptional landscapes links state to fate during differentiation. *Science* 367:eaaw3381.
- Setty M. *et al.* (2019) Characterization of cell fate probabilities in single-cell data with Palantir. *Nature Biotechnology* 37:451–460.

## ➡️ Next

So far every cell has been an isolated point in expression space; dissociation threw away where it lived. In
[Module 33 · Spatial omics: tissues as data](33_spatial_omics.ipynb) we keep the coordinates: spatially variable
genes, tissue domains, cell neighbourhoods and the unmixing of spots that contain several cells.